# EchoGesture: WLASL-2000 sign recognition on Kaggle

MediaPipe Holistic keypoints -> 64 x 576 features -> pure temporal 1D-CNN -> 2,000 classes ->
ONNX -> INT8 ONNX. Every stage reuses the project modules written into
`/kaggle/working/EchoGesture`. Stages run as subprocesses so the MediaPipe environment
(numpy 1.26) and the training environment (Kaggle PyTorch) never conflict.
Results only appear after the notebook is executed; nothing below is pre-filled.

## 1. Environment and hardware setup

In [ ]:
import csv
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.time()
INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT = WORK / "EchoGesture"
OUTPUT = WORK / "echogesture_output"
MP_VENV = WORK / "venv_mediapipe"
MP_PY = MP_VENV / "bin" / "python"
SYS_PY = Path(sys.executable)

SUBSET = "full"  # "full": all 21,095 videos; "reduced": documented 12,069-video subset
FRAME_STRIDE = 2  # 2: every 2nd frame (12.5 fps); 1: every frame (about 2x extraction time)
NUM_FRAMES = 64
ARCH = "tcnn"
EXTRACT_WORKERS = os.cpu_count() or 2
EPOCHS = 150
BATCH_SIZE = 64
PATIENCE = 20
SESSION_LIMIT_HOURS = 12
TRAINING_MINUTES = 120  # reserved for training when extraction runs in the same session
RESERVE_MINUTES = 40  # reserved for evaluation, export, benchmarking and saving
RUN_FULL_EXTRACTION = True
RUN_TRAINING = True
BENCHMARK_VIDEOS = 30

os.environ.update(
    {
        "ECHOGESTURE_WORK_DIR": str(PROJECT),
        "ECHOGESTURE_SUBSET": SUBSET,
        "ECHOGESTURE_FRAME_STRIDE": str(FRAME_STRIDE),
        "ECHOGESTURE_NUM_FRAMES": str(NUM_FRAMES),
        "ECHOGESTURE_ARCH": ARCH,
        "PYTHONPATH": str(PROJECT),
        "PYTHONUNBUFFERED": "1",
        "GLOG_minloglevel": "2",
    }
)
for directory in (PROJECT / "src", PROJECT / "kaggle", OUTPUT):
    directory.mkdir(parents=True, exist_ok=True)
KEYPOINT_DIR = PROJECT / "data" / "keypoints" / f"stride{FRAME_STRIDE}"
CHECKPOINT_DIR = PROJECT / "models" / "checkpoints" / ARCH
REPORT_DIR = PROJECT / "results" / "reports"
NOISE = (
    "W0000 ",
    "I0000 ",
    "All log messages before absl::InitializeLog",
    "Created TensorFlow Lite XNNPACK",
    "SymbolDatabase.GetPrototype",
    "warnings.warn(",
)


def run(*command, check: bool = True) -> int:
    """Run one pipeline stage in a subprocess and stream its progress output."""
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=PROJECT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    buffer = b""

    def emit(raw: bytes) -> None:
        line = raw.decode(errors="replace").rstrip()
        if line and not any(token in line for token in NOISE):
            print(line, flush=True)

    while chunk := process.stdout.read1(4096):
        *lines, buffer = re.split(rb"[\r\n]", buffer + chunk)
        for line in lines:
            emit(line)
    emit(buffer)
    code = process.wait()
    if check and code:
        raise RuntimeError(f"stage failed (exit {code}): {' '.join(map(str, command))}")
    return code


def minutes_left() -> float:
    return SESSION_LIMIT_HOURS * 60 - (time.time() - NOTEBOOK_START) / 60


def read_json(path: Path) -> dict:
    return json.loads(Path(path).read_text())


def resources() -> None:
    import psutil

    disk = shutil.disk_usage(WORK)
    memory = psutil.virtual_memory()
    print(
        f"disk free {disk.free / 2**30:.1f} GB | RAM available {memory.available / 2**30:.1f}"
        f" of {memory.total / 2**30:.1f} GB | session time left about {minutes_left():.0f} min"
    )

In [ ]:
import platform

import torch

print(f"Python {platform.python_version()} | PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()} | CUDA runtime: {torch.version.cuda}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(
        f"GPU {index}: {props.name} | {props.total_memory / 2**30:.1f} GB"
        f" | compute capability {props.major}.{props.minor}"
    )
if not torch.cuda.is_available():
    print("No CUDA GPU visible: training falls back to PyTorch CPU.")
print(f"CPU cores: {os.cpu_count()} (MediaPipe extraction is CPU-bound regardless of GPU)")
resources()

## 2. Dependency installation

MediaPipe 0.10.14 (the version used locally; legacy Holistic API) ships Linux wheels for
Python 3.9-3.12 and needs numpy < 2 and protobuf < 5, so it gets its own Python 3.11 venv
created with `uv`. Training uses Kaggle's preinstalled PyTorch. No TPU libraries are used.

In [ ]:
%%writefile {PROJECT}/kaggle/requirements-extraction.txt
numpy==1.26.4
mediapipe==0.10.14
opencv-python==4.10.0.84
opencv-contrib-python==4.11.0.86
protobuf==4.25.9
jax==0.7.1
jaxlib==0.7.1
pandas==3.0.6
psutil==7.2.2
onnxruntime==1.32.0
tqdm==4.70.1

In [ ]:
%%writefile {PROJECT}/kaggle/requirements-training.txt
onnx==1.22.0
onnxruntime==1.32.0
psutil
scikit-learn
matplotlib
pandas

In [ ]:
run(SYS_PY, "-m", "pip", "install", "--quiet", "uv", "-r", "kaggle/requirements-training.txt")
if not MP_PY.exists():
    run(SYS_PY, "-m", "uv", "venv", "--python", "3.11", MP_VENV)
run(
    SYS_PY,
    "-m",
    "uv",
    "pip",
    "install",
    "--quiet",
    "--python",
    MP_PY,
    "-r",
    "kaggle/requirements-extraction.txt",
)

In [ ]:
run(
    MP_PY,
    "-c",
    "import mediapipe as mp, numpy, cv2, pandas, onnxruntime, sys;"
    "mp.solutions.holistic.Holistic;"
    "print('MediaPipe env: Python', sys.version.split()[0], '| mediapipe', mp.__version__,"
    "'| numpy', numpy.__version__, '| opencv', cv2.__version__, '| pandas', pandas.__version__,"
    "'| onnxruntime', onnxruntime.__version__, '| legacy Holistic API available')",
)
run(
    SYS_PY,
    "-c",
    "import torch, onnx, onnxruntime, sklearn, numpy;"
    "print('Training env: torch', torch.__version__, '| onnx', onnx.__version__,"
    "'| onnxruntime', onnxruntime.__version__, '| scikit-learn', sklearn.__version__,"
    "'| numpy', numpy.__version__)",
)

## 3. Dataset discovery and validation

Writes the configuration and inspection modules, locates the mounted WLASL-2000 dataset under
`/kaggle/input`, validates it, and checks that the official splits and labels match the
manifest generated on the development machine.

In [ ]:
%%writefile {PROJECT}/src/__init__.py

In [ ]:
%%writefile {PROJECT}/src/config.py
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
# Environment overrides let the same code run on Kaggle (read-only input, writable working dir).
WORK_DIR = Path(os.environ.get("ECHOGESTURE_WORK_DIR", ROOT))

DATA_DIR = WORK_DIR / "data"
RAW_DIR = Path(os.environ.get("ECHOGESTURE_RAW_DIR", DATA_DIR / "raw" / "wlasl-complete"))
VIDEO_DIR = RAW_DIR / "videos"
MASTER_JSON = RAW_DIR / "WLASL_v0.3.json"
NSLT_JSON = RAW_DIR / "nslt_2000.json"
CLASS_LIST = RAW_DIR / "wlasl_class_list.txt"
# Every FRAME_STRIDE-th frame is extracted; stride 2 (12.5 fps) matched full-frame detection.
FRAME_STRIDE = int(os.environ.get("ECHOGESTURE_FRAME_STRIDE", 2))
KEYPOINT_DIR = Path(
    os.environ.get("ECHOGESTURE_KEYPOINT_DIR", DATA_DIR / "keypoints" / f"stride{FRAME_STRIDE}")
)
PROCESSED_DIR = DATA_DIR / "processed"
MANIFEST_CSV = PROCESSED_DIR / "manifest.csv"
REDUCED_MANIFEST_CSV = PROCESSED_DIR / "reduced_manifest.csv"
REDUCED_EXCLUDED_CSV = PROCESSED_DIR / "excluded_samples.csv"
# Final dataset: 10,000 videos (3 train / 1 val / 1 test per class) drawn from the reduced subset.
FINAL_MANIFEST_CSV = PROCESSED_DIR / "final_manifest.csv"
FINAL_EXCLUDED_CSV = PROCESSED_DIR / "final_excluded_samples.csv"
SUBSET_MANIFESTS = {
    "final": FINAL_MANIFEST_CSV,
    "reduced": REDUCED_MANIFEST_CSV,
    "full": MANIFEST_CSV,
}
TRAINING_SUBSET = os.environ.get("ECHOGESTURE_SUBSET", "final")
if TRAINING_SUBSET not in SUBSET_MANIFESTS:
    raise ValueError(f"ECHOGESTURE_SUBSET must be one of {sorted(SUBSET_MANIFESTS)}")
TRAINING_MANIFEST_CSV = SUBSET_MANIFESTS[TRAINING_SUBSET]

MODEL_DIR = WORK_DIR / "models"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
EXPORT_DIR = MODEL_DIR / "exported"
DEFAULT_ARCH = os.environ.get("ECHOGESTURE_ARCH", "tcnn")

RESULTS_DIR = WORK_DIR / "results"
PLOT_DIR = RESULTS_DIR / "plots"
REPORT_DIR = RESULTS_DIR / "reports"
RESULTS_CSV = RESULTS_DIR / "results.csv"

NUM_CLASSES = 2000
SPLITS = ("train", "val", "test")
SEED = 42

NUM_POSE = 33
NUM_HAND = 21
NUM_LANDMARKS = NUM_POSE + 2 * NUM_HAND
LEFT_HAND_SLICE = slice(NUM_POSE, NUM_POSE + NUM_HAND)
RIGHT_HAND_SLICE = slice(NUM_POSE + NUM_HAND, NUM_LANDMARKS)
COORDS = 3

LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_WRIST = 15
RIGHT_WRIST = 16

# Hand identity follows the signer's anatomy on unmirrored video, as reported by MediaPipe.
LANDMARK_ORDER = ("pose", "left_hand", "right_hand")

EXTRACTION_STATUS_CSV = KEYPOINT_DIR / "extraction_status.csv"
HOLISTIC_MODEL_COMPLEXITY = 1
HOLISTIC_MIN_DETECTION_CONFIDENCE = 0.5
HOLISTIC_MIN_TRACKING_CONFIDENCE = 0.5

# Baseline 64 frames per the original specification; 32 was an unvalidated experiment.
NUM_FRAMES = int(os.environ.get("ECHOGESTURE_NUM_FRAMES", 64))
# Interpolate gaps up to 0.32 s (8 frames at 25 fps).
MAX_FILL_GAP = 8 // FRAME_STRIDE
CLIP_VALUE = 6.0
MIN_SHOULDER_WIDTH = 0.02
# Hips, legs and feet (23-32) are mostly off-frame in WLASL and carry no sign information.
DROPPED_POSE = tuple(range(23, 33))
POSE_MIRROR_PAIRS = (
    (1, 4),
    (2, 5),
    (3, 6),
    (7, 8),
    (9, 10),
    (11, 12),
    (13, 14),
    (15, 16),
    (17, 18),
    (19, 20),
    (21, 22),
    (23, 24),
    (25, 26),
    (27, 28),
    (29, 30),
    (31, 32),
)
PREPROCESS_CONFIG_JSON = PROCESSED_DIR / "preprocess_config.json"


def ensure_dirs() -> None:
    for path in (
        KEYPOINT_DIR,
        PROCESSED_DIR,
        CHECKPOINT_DIR,
        EXPORT_DIR,
        PLOT_DIR,
        REPORT_DIR,
    ):
        path.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile {PROJECT}/src/inspect_dataset.py
import argparse
import json
import logging
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2
import pandas as pd

from src import config

log = logging.getLogger("inspect_dataset")


def load_class_list(path: Path) -> dict[int, str]:
    classes = {}
    for line in path.read_text().splitlines():
        if line.strip():
            index, gloss = line.split("\t", 1)
            classes[int(index)] = gloss.strip()
    return classes


def load_master(path: Path) -> tuple[pd.DataFrame, list[str]]:
    entries = json.loads(path.read_text())
    rows = []
    for gloss_index, entry in enumerate(entries):
        for inst in entry["instances"]:
            rows.append(
                {
                    "video_id": inst["video_id"],
                    "gloss": entry["gloss"],
                    "master_label": gloss_index,
                    "master_split": inst["split"],
                    "signer_id": inst["signer_id"],
                    "fps": inst["fps"],
                    "frame_start": inst["frame_start"],
                    "frame_end": inst["frame_end"],
                }
            )
    return pd.DataFrame(rows), [e["gloss"] for e in entries]


def load_nslt(path: Path) -> pd.DataFrame:
    data = json.loads(path.read_text())
    rows = [
        {
            "video_id": vid,
            "split": info["subset"],
            "label": info["action"][0],
            "nslt_start": info["action"][1],
            "nslt_end": info["action"][2],
        }
        for vid, info in data.items()
    ]
    return pd.DataFrame(rows)


def probe_video(path: Path) -> dict:
    result = {"exists": path.is_file(), "readable": False, "num_frames": 0}
    result.update(width=0, height=0, video_fps=0.0)
    if not result["exists"]:
        return result
    cap = cv2.VideoCapture(str(path))
    try:
        ok, frame = cap.read()
        result["readable"] = bool(ok and frame is not None)
        result["num_frames"] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        result["width"] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        result["height"] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        result["video_fps"] = float(cap.get(cv2.CAP_PROP_FPS))
    finally:
        cap.release()
    return result


def probe_all(video_ids: list[str], workers: int) -> pd.DataFrame:
    paths = [config.VIDEO_DIR / f"{vid}.mp4" for vid in video_ids]
    with ProcessPoolExecutor(max_workers=workers) as pool:
        probes = list(pool.map(probe_video, paths, chunksize=64))
    return pd.DataFrame(probes).assign(video_id=video_ids)


def exclusion_reason(row: pd.Series) -> str | None:
    if pd.isna(row["split"]):
        return "missing_nslt_annotation"
    if not row["exists"]:
        return "missing_video"
    if not row["readable"]:
        return "unreadable_video"
    if row["num_frames"] < 2:
        return "too_few_frames"
    return None


def resolve_clip_bounds(df: pd.DataFrame, tolerance: int = 2) -> pd.DataFrame:
    # Most resized videos are pre-trimmed; trim only when the NSLT segment lies strictly inside.
    segment = df["nslt_end"] - df["nslt_start"] + 1
    inside = (df["nslt_end"] <= df["num_frames"]) & ((segment - df["num_frames"]).abs() > tolerance)
    df["clip_start"] = (df["nslt_start"] - 1).where(inside, 0).astype(int)
    df["clip_end"] = df["nslt_end"].where(inside, df["num_frames"]).astype(int)
    return df


def build_report(df: pd.DataFrame, classes: dict[int, str], master_glosses: list[str]) -> dict:
    valid = df[df["excluded_reason"].isna()]
    per_class = valid.groupby("label").size().reindex(range(config.NUM_CLASSES), fill_value=0)
    split_counts = {s: int((valid["split"] == s).sum()) for s in config.SPLITS}
    train_per_class = (
        valid[valid["split"] == "train"]
        .groupby("label")
        .size()
        .reindex(range(config.NUM_CLASSES), fill_value=0)
    )
    disk_ids = {p.stem for p in config.VIDEO_DIR.glob("*.mp4")}
    return {
        "annotated_instances": int(len(df)),
        "videos_on_disk": len(disk_ids),
        "videos_on_disk_without_annotation": len(disk_ids - set(df["video_id"])),
        "valid_videos": int(len(valid)),
        "excluded": dict(Counter(df["excluded_reason"].dropna())),
        "num_classes_class_list": len(classes),
        "num_classes_master": len(master_glosses),
        "class_list_matches_master": all(classes.get(i) == g for i, g in enumerate(master_glosses)),
        "nslt_label_matches_master": bool(
            (
                df.dropna(subset=["label"])["label"] == df.dropna(subset=["label"])["master_label"]
            ).all()
        ),
        "nslt_split_matches_master": bool(
            (
                df.dropna(subset=["split"])["split"] == df.dropna(subset=["split"])["master_split"]
            ).all()
        ),
        "classes_with_valid_samples": int((per_class > 0).sum()),
        "classes_without_train_samples": int((train_per_class == 0).sum()),
        "classes_without_test_samples": int(
            config.NUM_CLASSES - valid[valid["split"] == "test"]["label"].nunique()
        ),
        "split_counts": split_counts,
        "samples_per_class": {
            "min": int(per_class.min()),
            "max": int(per_class.max()),
            "mean": round(float(per_class.mean()), 2),
            "median": float(per_class.median()),
        },
        "imbalance_ratio_max_min": round(float(per_class.max() / max(per_class.min(), 1)), 2),
        "unique_signers": int(valid["signer_id"].nunique()),
        "frames": valid["num_frames"].describe().round(1).to_dict(),
        "resolutions": {
            f"{w}x{h}": int(n)
            for (w, h), n in valid.groupby(["width", "height"]).size().nlargest(5).items()
        },
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--workers", type=int, default=4)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    config.ensure_dirs()

    classes = load_class_list(config.CLASS_LIST)
    master, master_glosses = load_master(config.MASTER_JSON)
    nslt = load_nslt(config.NSLT_JSON)

    duplicates = master[master["video_id"].duplicated(keep=False)]
    if not duplicates.empty:
        raise ValueError(
            f"Duplicate video IDs in master JSON: {sorted(duplicates['video_id'])[:10]}"
        )
    nslt_only = sorted(set(nslt["video_id"]) - set(master["video_id"]))
    master_only = sorted(set(master["video_id"]) - set(nslt["video_id"]))

    # NSLT-2000 defines the official splits and labels; master JSON adds metadata.
    df = nslt.merge(master, on="video_id", how="outer")
    df["gloss"] = df["gloss"].fillna(df["label"].map(lambda i: master_glosses[int(i)]))
    df["master_label"] = df["master_label"].fillna(df["label"])
    df["master_split"] = df["master_split"].fillna(df["split"])
    df["signer_id"] = df["signer_id"].fillna(-1).astype(int)
    df = df.merge(probe_all(df["video_id"].tolist(), args.workers), on="video_id")
    df["excluded_reason"] = df.apply(exclusion_reason, axis=1)

    report = build_report(df, classes, master_glosses)
    report["duplicate_video_ids"] = int(len(duplicates))
    report["nslt_ids_missing_from_master"] = nslt_only
    report["master_ids_missing_from_nslt"] = master_only
    report["class_list_mismatches"] = {
        i: {"class_list": classes.get(i), "master": g}
        for i, g in enumerate(master_glosses)
        if classes.get(i) != g
    }

    df[df["excluded_reason"].notna()][
        ["video_id", "gloss", "master_split", "excluded_reason"]
    ].to_csv(config.REPORT_DIR / "excluded_samples.csv", index=False)

    valid = df[df["excluded_reason"].isna()].copy()
    valid["label"] = valid["label"].astype(int)
    valid = resolve_clip_bounds(valid)
    report["clips_trimmed_to_nslt_segment"] = int(
        ((valid["clip_start"] > 0) | (valid["clip_end"] < valid["num_frames"])).sum()
    )
    valid.groupby(["label", "gloss", "split"]).size().unstack(fill_value=0).to_csv(
        config.REPORT_DIR / "class_counts.csv"
    )
    valid[
        [
            "video_id",
            "label",
            "gloss",
            "split",
            "signer_id",
            "num_frames",
            "video_fps",
            "clip_start",
            "clip_end",
            "width",
            "height",
        ]
    ].to_csv(config.MANIFEST_CSV, index=False)

    (config.REPORT_DIR / "dataset_report.json").write_text(json.dumps(report, indent=2))
    log.info(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {PROJECT}/src/validate_keypoints.py
import argparse
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from src import config

log = logging.getLogger("validate_keypoints")

GROUPS = {
    "pose": slice(0, config.NUM_POSE),
    "left_hand": config.LEFT_HAND_SLICE,
    "right_hand": config.RIGHT_HAND_SLICE,
}


def expected_frames(clip_start: int, clip_end: int, stride: int = config.FRAME_STRIDE) -> int:
    return len(range(clip_start, clip_end, stride))


def array_problems(keypoints: np.ndarray, expected: int, status_frames: int | None) -> list[str]:
    if keypoints.dtype != np.float32 or keypoints.shape[1:] != (
        config.NUM_LANDMARKS,
        config.COORDS,
    ):
        return [f"shape/dtype {keypoints.shape} {keypoints.dtype}"]
    problems = []
    if not 0 < len(keypoints) <= expected:
        problems.append(f"{len(keypoints)} frames, expected 1..{expected}")
    if status_frames is not None and len(keypoints) != status_frames:
        problems.append(f"{len(keypoints)} frames, status records {status_frames}")
    if np.isinf(keypoints).any():
        problems.append("infinite values")
    for group, part in GROUPS.items():
        nan = np.isnan(keypoints[:, part])
        if not (nan.all(axis=(1, 2)) | ~nan.any(axis=(1, 2))).all():
            problems.append(f"partially missing {group}")
    return problems


def wrist_distances(keypoints: np.ndarray, hand_wrist: int, pose_wrist: int) -> np.ndarray:
    both = ~np.isnan(keypoints[:, hand_wrist, 0]) & ~np.isnan(keypoints[:, pose_wrist, 0])
    return np.linalg.norm(
        keypoints[both, hand_wrist, :2] - keypoints[both, pose_wrist, :2], axis=-1
    )


def ordering_summary(arrays: list[np.ndarray]) -> dict:
    # Each hand's wrist must sit nearer the same-side pose wrist than the opposite one.
    pairs = {
        "left_hand": (config.LEFT_HAND_SLICE.start, config.LEFT_WRIST, config.RIGHT_WRIST),
        "right_hand": (config.RIGHT_HAND_SLICE.start, config.RIGHT_WRIST, config.LEFT_WRIST),
    }
    summary = {}
    for name, (hand, same, other) in pairs.items():
        same_d = np.concatenate([wrist_distances(k, hand, same) for k in arrays])
        other_d = np.concatenate([wrist_distances(k, hand, other) for k in arrays])
        if not len(same_d):
            summary[name] = {"testable": False}
            continue
        summary[name] = {
            "median_same_side": float(np.median(same_d)),
            "median_opposite_side": float(np.median(other_d)),
            "ordering_ok": bool(np.median(same_d) < np.median(other_d)),
        }
    return summary


def validate(manifest: pd.DataFrame, keypoint_dir: Path, ordering_sample: int = 2000) -> dict:
    status_path = keypoint_dir / config.EXTRACTION_STATUS_CSV.name
    status = pd.read_csv(status_path, dtype={"video_id": str}, keep_default_na=False)
    status = status.drop_duplicates("video_id", keep="last").set_index("video_id")
    arrays_on_disk = {p.stem for p in keypoint_dir.glob("*.npy")}
    rows, ordering_arrays, missing_hand_frames, frames = [], [], 0, 0
    for row in manifest.itertuples():
        state = status["status"].get(row.video_id, "pending")
        record = {"video_id": row.video_id, "split": row.split, "label": row.label}
        if state == "failed":
            rows.append(record | {"state": "failed", "detail": status.loc[row.video_id, "error"]})
            continue
        if state != "ok":
            rows.append(record | {"state": "pending", "detail": ""})
            continue
        if row.video_id not in arrays_on_disk:
            rows.append(record | {"state": "corrupted", "detail": "status ok but array missing"})
            continue
        try:
            keypoints = np.load(keypoint_dir / f"{row.video_id}.npy")
        except (OSError, ValueError) as exc:
            rows.append(record | {"state": "corrupted", "detail": f"unreadable: {exc!r}"})
            continue
        problems = array_problems(
            keypoints,
            expected_frames(row.clip_start, row.clip_end),
            int(status.loc[row.video_id, "frames"]),
        )
        if problems:
            rows.append(record | {"state": "corrupted", "detail": "; ".join(problems)})
            continue
        rows.append(record | {"state": "completed", "detail": ""})
        frames += len(keypoints)
        hands = np.isnan(keypoints[:, config.NUM_POSE :, 0]).reshape(len(keypoints), 2, -1)
        missing_hand_frames += int(hands.all(axis=-1).sum())
        if len(ordering_arrays) < ordering_sample:
            ordering_arrays.append(keypoints)
    table = pd.DataFrame(rows)
    counts = table.groupby(["split", "state"]).size().unstack(fill_value=0)
    completed = table[table["state"] == "completed"]
    orphans = sorted(arrays_on_disk - set(status.index[status["status"] == "ok"]))
    return {
        "table": table,
        "report": {
            "manifest_videos": len(manifest),
            "counts": table["state"]
            .value_counts()
            .reindex(["completed", "pending", "failed", "corrupted"], fill_value=0)
            .to_dict(),
            "counts_by_split": counts.to_dict(orient="index"),
            "classes_with_completed_video": {
                split: int(completed[completed["split"] == split]["label"].nunique())
                for split in config.SPLITS
            },
            "arrays_without_ok_status": len(orphans),
            "temp_files": len(list(keypoint_dir.glob(".*tmp*"))),
            "frames_in_completed_arrays": frames,
            "missing_hand_slots_fraction": missing_hand_frames / max(2 * frames, 1),
            "landmark_ordering": ordering_summary(ordering_arrays) if ordering_arrays else {},
            "frame_stride": config.FRAME_STRIDE,
            "keypoint_dir": str(keypoint_dir),
        },
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--manifest", type=Path, default=config.TRAINING_MANIFEST_CSV)
    parser.add_argument("--keypoint-dir", type=Path, default=config.KEYPOINT_DIR)
    parser.add_argument("--name", default=None)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    config.ensure_dirs()

    manifest = pd.read_csv(args.manifest, dtype={"video_id": str})
    result = validate(manifest, args.keypoint_dir)
    name = args.name or args.manifest.stem
    report = result["report"] | {"manifest": args.manifest.name}
    problems = result["table"][result["table"]["state"].isin(["failed", "corrupted"])]
    problems.to_csv(config.REPORT_DIR / f"keypoint_problems_{name}.csv", index=False)
    (config.REPORT_DIR / f"keypoint_validation_{name}.json").write_text(
        json.dumps(report, indent=2)
    )
    log.info(json.dumps(report, indent=2))
    ordering_ok = all(v.get("ordering_ok", True) for v in report["landmark_ordering"].values())
    sys.exit(0 if report["counts"]["corrupted"] == 0 and ordering_ok else 1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {PROJECT}/kaggle/reference.json
{
  "manifest": {
    "rows": 21095,
    "sha256": "0618356a70c2edb89f49362a155af6f9fef11597295c008707bfde822d293ce0"
  },
  "reduced_manifest": {
    "rows": 12069,
    "sha256": "9711f4c8889d4e061d22da68473ee5e10c8c800a1d3ac97b8a880a731d9e4c00"
  },
  "frame_stride": 2,
  "mediapipe": "0.10.14",
  "platform": "macOS arm64 (Apple M4)",
  "reference_videos": [
    {
      "video_id": "61062",
      "clip_start": 0,
      "clip_end": 37,
      "frames": 19,
      "detection_rate": {
        "pose": 1.0,
        "left_hand": 1.0,
        "right_hand": 1.0
      },
      "mean_xy": {
        "pose": [
          0.4944300055503845,
          0.719789981842041
        ],
        "left_hand": [
          0.38284000754356384,
          0.6786900162696838
        ],
        "right_hand": [
          0.43309998512268066,
          0.5298799872398376
        ]
      }
    },
    {
      "video_id": "23450",
      "clip_start": 0,
      "clip_end": 70,
      "frames": 35,
      "detection_rate": {
        "pose": 1.0,
        "left_hand": 0.5428571428571428,
        "right_hand": 0.5428571428571428
      },
      "mean_xy": {
        "pose": [
          0.5082899928092957,
          0.7777000069618225
        ],
        "left_hand": [
          0.6171900033950806,
          0.5568900108337402
        ],
        "right_hand": [
          0.4124700129032135,
          0.5368599891662598
        ]
      }
    },
    {
      "video_id": "17607",
      "clip_start": 0,
      "clip_end": 76,
      "frames": 38,
      "detection_rate": {
        "pose": 1.0,
        "left_hand": 0.5263157894736842,
        "right_hand": 0.5263157894736842
      },
      "mean_xy": {
        "pose": [
          0.5059099793434143,
          0.7846400141716003
        ],
        "left_hand": [
          0.5643399953842163,
          0.4996500015258789
        ],
        "right_hand": [
          0.43393999338150024,
          0.5144500136375427
        ]
      }
    }
  ]
}

In [ ]:
%%writefile {PROJECT}/kaggle/kaggle_checks.py
import argparse
import hashlib
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from src import config
from src.validate_keypoints import GROUPS, array_problems, expected_frames, ordering_summary

REFERENCE = Path(__file__).with_name("reference.json")


def split_hash(path: Path) -> dict:
    manifest = pd.read_csv(path, dtype={"video_id": str})
    rows = sorted(
        f"{v},{s},{label}"
        for v, s, label in zip(manifest.video_id, manifest.split, manifest.label, strict=True)
    )
    return {"rows": len(rows), "sha256": hashlib.sha256("\n".join(rows).encode()).hexdigest()}


MANIFESTS = {"manifest": config.MANIFEST_CSV, "reduced_manifest": config.REDUCED_MANIFEST_CSV}


def check_manifests(keys: list[str]) -> bool:
    reference = json.loads(REFERENCE.read_text())
    ok = True
    for key in keys:
        path = MANIFESTS[key]
        actual = split_hash(path)
        match = actual == reference[key]
        ok &= match
        print(
            f"{key}: {actual['rows']} rows, splits/labels "
            f"{'MATCH' if match else 'DIFFER'} reference"
        )
    return ok


def check_parity(rate_tolerance: float, xy_tolerance: float) -> bool:
    from src.extract_keypoints import create_holistic, detection_rates, extract_video

    reference = json.loads(REFERENCE.read_text())
    ok = True
    for ref in reference["reference_videos"]:
        with create_holistic() as holistic:
            keypoints = extract_video(
                holistic,
                config.VIDEO_DIR / f"{ref['video_id']}.mp4",
                ref["clip_start"],
                ref["clip_end"],
                reference["frame_stride"],
            )
        rates = dict(zip(GROUPS, detection_rates(keypoints), strict=True))
        frames_ok = len(keypoints) == ref["frames"]
        print(f"video {ref['video_id']}: frames {len(keypoints)} (reference {ref['frames']})")
        ok &= frames_ok
        for group, part in GROUPS.items():
            rate_diff = abs(rates[group] - ref["detection_rate"][group])
            xy = np.nanmean(keypoints[:, part, :2], axis=(0, 1))
            xy_diff = float(np.nanmax(np.abs(xy - np.asarray(ref["mean_xy"][group]))))
            group_ok = rate_diff <= rate_tolerance and (
                math.isnan(xy_diff) or xy_diff <= xy_tolerance
            )
            ok &= group_ok
            print(
                f"  {group:10s} detection {rates[group]:.3f} vs {ref['detection_rate'][group]:.3f}"
                f"  mean-xy diff {xy_diff:.4f}  {'ok' if group_ok else 'OUT OF TOLERANCE'}"
            )
    return ok


def check_ordering(arrays: list[np.ndarray]) -> bool:
    ok = True
    for name, result in ordering_summary(arrays).items():
        if not result.get("median_same_side"):
            print(f"  {name}: not detected in sample, ordering not testable")
            continue
        ok &= result["ordering_ok"]
        print(
            f"  {name}: median distance to same-side pose wrist {result['median_same_side']:.3f}, "
            f"opposite {result['median_opposite_side']:.3f} -> "
            f"{'ordering ok' if result['ordering_ok'] else 'SWAPPED'}"
        )
    return ok


def check_samples(directory: Path) -> bool:
    manifest = pd.read_csv(config.MANIFEST_CSV, dtype={"video_id": str}).set_index("video_id")
    files = sorted(directory.glob("*.npy"))
    arrays = [np.load(path) for path in files]
    problems = []
    missing_hand_frames = 0
    for path, keypoints in zip(files, arrays, strict=True):
        row = manifest.loc[path.stem]
        expected = expected_frames(row.clip_start, row.clip_end)
        problems += [f"{path.stem}: {p}" for p in array_problems(keypoints, expected, None)]
        hands = np.isnan(keypoints[:, config.NUM_POSE :, 0]).reshape(len(keypoints), 2, -1)
        missing_hand_frames += int(hands.all(axis=-1).sum())
    frames = sum(len(k) for k in arrays)
    print(f"{len(files)} arrays, {frames} frames, per-frame shape (75, 3), float32 verified")
    print(f"missing-hand frames stored as all-NaN groups: {missing_hand_frames}")
    ordering_ok = check_ordering(arrays)
    print(f"{len(problems)} problems")
    for problem in problems:
        print("  " + problem)
    return bool(files) and not problems and ordering_ok


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="command", required=True)
    manifests = sub.add_parser("manifests")
    manifests.add_argument("keys", nargs="+", choices=sorted(MANIFESTS))
    parity = sub.add_parser("parity")
    parity.add_argument("--rate-tolerance", type=float, default=0.2)
    parity.add_argument("--xy-tolerance", type=float, default=0.03)
    samples = sub.add_parser("samples")
    samples.add_argument("directory", type=Path)
    args = parser.parse_args()
    if args.command == "manifests":
        ok = check_manifests(args.keys)
    elif args.command == "parity":
        ok = check_parity(args.rate_tolerance, args.xy_tolerance)
    else:
        ok = check_samples(args.directory)
    sys.exit(0 if ok else 1)


if __name__ == "__main__":
    main()

In [ ]:
EXPECTED_HASHES = {
    "src/__init__.py": "e3b0c44298fc1c14",
    "src/config.py": "145c7e644c20a8fa",
    "src/inspect_dataset.py": "ddb1987f75fcb0bf",
    "src/select_subset.py": "de36ae04af042da9",
    "src/extract_keypoints.py": "271c466d41596e53",
    "src/preprocess.py": "d2067496ff3f8d60",
    "src/augment.py": "4f28e70464ac10a7",
    "src/dataset.py": "a8495cad70095b17",
    "src/model.py": "0c9b9343e479fe4c",
    "src/train.py": "8e048539b03a3c64",
    "src/evaluate.py": "15bd1c2559d8525b",
    "src/export_quantize.py": "c19320ec107bdf59",
    "src/inference.py": "d771a0d6754b4e94",
    "src/benchmark.py": "89b0cdd2404cdad2",
    "src/validate_keypoints.py": "043af524e92d74e7",
    "main.py": "19f2fbe60619ce6b",
    "kaggle/kaggle_checks.py": "3f79fa00bc4867e7",
    "kaggle/reference.json": "f98238ddc97b1728",
    "kaggle/requirements-extraction.txt": "012adf406d8d7246",
    "kaggle/requirements-training.txt": "67f588b44f839140",
}
packages = [p.parent for p in INPUT_ROOT.rglob("package_hashes.json")]
for package in packages:
    stored = read_json(package / "package_hashes.json")
    differing = [name for name, digest in stored.items() if digest != EXPECTED_HASHES.get(name)]
    status = "identical to this notebook" if not differing else f"DIFFERS in {differing}"
    print(f"Attached package {package}: {status}")
if not packages:
    print("No EchoGesture package attached (optional); using the modules written by this notebook.")

In [ ]:
def find_dirs(marker: str) -> list[Path]:
    found = []
    for root, dirs, files in os.walk(INPUT_ROOT, followlinks=True):
        dirs[:] = [d for d in dirs if d != "videos"]
        if marker in files:
            found.append(Path(root))
    return found


required = ("WLASL_v0.3.json", "nslt_2000.json", "wlasl_class_list.txt")
candidates = [
    d
    for d in find_dirs("nslt_2000.json")
    if (d / "videos").is_dir() and all((d / name).exists() for name in required)
]
if len(candidates) != 1:
    raise RuntimeError(
        f"Expected one WLASL-2000 directory under {INPUT_ROOT}, found {candidates}. "
        "Attach the WLASL2000 Resized dataset (see KAGGLE_README.md)."
    )
RAW_DIR = candidates[0]
os.environ["ECHOGESTURE_RAW_DIR"] = str(RAW_DIR)
video_count = sum(1 for _ in (RAW_DIR / "videos").glob("*.mp4"))
print(f"Dataset directory: {RAW_DIR}")
print(f"Annotation files: {', '.join(required)} | videos: {video_count}")

In [ ]:
run(MP_PY, "-m", "src.inspect_dataset", "--workers", EXTRACT_WORKERS)
report = read_json(REPORT_DIR / "dataset_report.json")
for key in (
    "videos_on_disk",
    "valid_videos",
    "excluded",
    "num_classes_master",
    "classes_with_valid_samples",
    "classes_without_test_samples",
    "split_counts",
    "samples_per_class",
    "duplicate_video_ids",
    "nslt_ids_missing_from_master",
    "class_list_mismatches",
):
    print(f"{key}: {report[key]}")
run(MP_PY, "kaggle/kaggle_checks.py", "manifests", "manifest")

## 4. Manifest inspection

The reduced manifest (12,069 videos) was created only to fit a 60-minute local run: it halves
training clips per class (minimum 3), keeps one validation clip per class and keeps every
official test clip. `SUBSET = "full"` (default) trains on all 21,095 videos.

In [ ]:
%%writefile {PROJECT}/src/select_subset.py
import argparse
import json
import logging

import numpy as np
import pandas as pd

from src import config

log = logging.getLogger("select_subset")

SPLIT_FRACTIONS = {"train": 0.5, "val": 0.0, "test": 1.0}
SPLIT_MINIMUMS = {"train": 3, "val": 1, "test": 1}
FINAL_PER_CLASS = {"train": 3, "val": 1, "test": 1}


def signer_diverse_order(group: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    # Round-robin over signers so earlier picks cover as many distinct signers as possible.
    shuffled = group.iloc[rng.permutation(len(group))]
    rank_within_signer = shuffled.groupby("signer_id").cumcount()
    return shuffled.assign(_rank=rank_within_signer.values).sort_values("_rank", kind="stable")


def quota(available: int, fraction: float, minimum: int) -> int:
    return min(available, max(minimum, int(available * fraction)))


def select(
    manifest: pd.DataFrame, fractions: dict[str, float], minimums: dict[str, int], seed: int
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    chosen = []
    for (split, _), group in manifest.groupby(["split", "label"], sort=True):
        ordered = signer_diverse_order(group, rng)
        take = quota(len(group), fractions[split], minimums[split])
        chosen.append(ordered.head(take).drop(columns="_rank").assign(class_rank=range(take)))
    selected = pd.concat(chosen)
    # Extraction priority: every class's first sample before any class's second sample.
    order = rng.permutation(len(selected))
    selected = selected.assign(_tiebreak=order).sort_values(["class_rank", "_tiebreak"])
    return selected.drop(columns="_tiebreak").reset_index(drop=True).rename_axis("priority")


def shortfalls(
    manifest: pd.DataFrame, selected: pd.DataFrame, minimums: dict[str, int]
) -> list[dict]:
    rows = []
    for split, minimum in minimums.items():
        counts = (
            selected[selected["split"] == split]
            .groupby("label")
            .size()
            .reindex(range(config.NUM_CLASSES), fill_value=0)
        )
        for label, count in counts[counts < minimum].items():
            rows.append({"split": split, "label": int(label), "selected": int(count)})
    return rows


def summary(manifest: pd.DataFrame, selected: pd.DataFrame, minimums: dict) -> dict:
    report = {
        "strategy": {
            "fractions": SPLIT_FRACTIONS,
            "minimum_per_class": minimums,
            "method": (
                "per split and class: floor(n * fraction) clamped to [minimum, n], "
                "signer round-robin, seeded shuffle"
            ),
            "seed": config.SEED,
        }
    }
    for split in config.SPLITS:
        orig = manifest[manifest["split"] == split]
        sel = selected[selected["split"] == split]
        per_class = sel.groupby("label").size()
        report[split] = {
            "original": int(len(orig)),
            "selected": int(len(sel)),
            "classes": int(sel["label"].nunique()),
            "per_class_min": int(per_class.min()),
            "per_class_median": float(per_class.median()),
            "per_class_max": int(per_class.max()),
            "signers_original": int(orig["signer_id"].nunique()),
            "signers_selected": int(sel["signer_id"].nunique()),
        }
    report["total_selected"] = int(len(selected))
    report["classes_below_minimum"] = shortfalls(manifest, selected, minimums)
    return report


def select_final(reduced: pd.DataFrame, per_class: dict[str, int]) -> pd.DataFrame:
    # First k signer-diverse clips per split and class, using the reduced manifest's fixed ranks.
    keep = reduced["class_rank"] < reduced["split"].map(per_class)
    return reduced[keep].sort_values("priority").reset_index(drop=True)


def final_exclusions(manifest: pd.DataFrame, reduced: pd.DataFrame, final: pd.DataFrame):
    excluded = manifest[~manifest["video_id"].isin(final["video_id"])].copy()
    in_reduced = excluded["video_id"].isin(reduced["video_id"])
    excluded["reason"] = np.where(
        in_reduced, "over_final_per_class_quota", "not_selected_subset_reduction"
    )
    return excluded[["video_id", "label", "gloss", "split", "signer_id", "reason"]]


def write_final(manifest: pd.DataFrame) -> dict:
    reduced = pd.read_csv(config.REDUCED_MANIFEST_CSV, dtype={"video_id": str})
    final = select_final(reduced, FINAL_PER_CLASS)
    expected = config.NUM_CLASSES * sum(FINAL_PER_CLASS.values())
    if len(final) != expected:
        raise ValueError(f"final subset has {len(final)} videos, expected {expected}")
    final.to_csv(config.FINAL_MANIFEST_CSV, index=False)
    excluded = final_exclusions(manifest, reduced, final)
    excluded.to_csv(config.FINAL_EXCLUDED_CSV, index=False)
    report = summary(manifest, final, FINAL_PER_CLASS)
    report["strategy"] = {
        "source": config.REDUCED_MANIFEST_CSV.name,
        "per_class": FINAL_PER_CLASS,
        "method": "first k clips per split and class by the reduced manifest's class_rank "
        "(signer round-robin, seed 42); official splits unchanged",
    }
    report["excluded_reasons"] = excluded["reason"].value_counts().to_dict()
    report["excluded_by_split"] = excluded.groupby("split").size().to_dict()
    return report


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--seed", type=int, default=config.SEED)
    parser.add_argument("--final", action="store_true", help="derive the 10,000-video subset")
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    if args.final:
        manifest = pd.read_csv(config.MANIFEST_CSV, dtype={"video_id": str})
        report = write_final(manifest)
        (config.REPORT_DIR / "final_selection_report.json").write_text(json.dumps(report, indent=2))
        log.info(json.dumps(report, indent=2))
        return

    manifest = pd.read_csv(config.MANIFEST_CSV, dtype={"video_id": str})
    selected = select(manifest, SPLIT_FRACTIONS, SPLIT_MINIMUMS, args.seed)
    excluded = manifest[~manifest["video_id"].isin(selected["video_id"])].assign(
        reason="not_selected_subset_reduction"
    )
    selected.to_csv(config.REDUCED_MANIFEST_CSV)
    excluded[["video_id", "label", "gloss", "split", "signer_id", "reason"]].to_csv(
        config.REDUCED_EXCLUDED_CSV, index=False
    )
    report = summary(manifest, selected, SPLIT_MINIMUMS)
    (config.REPORT_DIR / "selection_report.json").write_text(json.dumps(report, indent=2))
    log.info(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

run(MP_PY, "-m", "src.select_subset")
run(MP_PY, "kaggle/kaggle_checks.py", "manifests", "reduced_manifest")
processed = PROJECT / "data" / "processed"
full = pd.read_csv(processed / "manifest.csv", dtype={"video_id": str})
reduced = pd.read_csv(processed / "reduced_manifest.csv", dtype={"video_id": str})
excluded = pd.read_csv(processed / "excluded_samples.csv", dtype={"video_id": str})
summary = (
    pd.DataFrame(
        {
            "full": full.groupby("split").size(),
            "reduced": reduced.groupby("split").size(),
            "excluded": excluded.groupby("split").size(),
            "classes_full": full.groupby("split")["label"].nunique(),
            "classes_reduced": reduced.groupby("split")["label"].nunique(),
        }
    )
    .fillna(0)
    .astype(int)
)
display(summary)
print("Exclusion reasons:", excluded["reason"].value_counts().to_dict())
print("Official test clips removed by the reduction:", int(summary.loc["test", "excluded"]))
print(
    "Reduced train clips per class:",
    reduced[reduced.split == "train"].groupby("label").size().describe().round(2).to_dict(),
)
print(
    f"SUBSET = {SUBSET!r}: training uses {len(full) if SUBSET == 'full' else len(reduced)} videos"
)

## 5. MediaPipe extraction

33 pose + 21 left-hand + 21 right-hand landmarks, (x, y, z) each, saved per video as
`(T, 75, 3)` float32 with missing groups as NaN. Extraction is resumable: arrays are written
atomically, every video gets a status row, and finished videos are skipped. Previous progress
attached as a Kaggle input (a `stride2` keypoint folder or a checkpoint folder) is restored
first. MediaPipe runs on CPU; a GPU does not speed it up.

In [ ]:
%%writefile {PROJECT}/src/extract_keypoints.py
import argparse
import csv
import json
import logging
import os
import time
import warnings
from collections.abc import Iterator
from dataclasses import asdict, dataclass
from multiprocessing import Pool
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import psutil
from tqdm import tqdm

from src import config

log = logging.getLogger("extract_keypoints")

STATUS_FIELDS = [
    "video_id",
    "status",
    "frames",
    "pose_rate",
    "left_rate",
    "right_rate",
    "seconds",
    "error",
]


@dataclass
class ExtractionResult:
    video_id: str
    status: str
    frames: int = 0
    pose_rate: float = 0.0
    left_rate: float = 0.0
    right_rate: float = 0.0
    seconds: float = 0.0
    error: str = ""


def landmarks_to_array(landmarks, count: int) -> np.ndarray:
    if landmarks is None:
        return np.full((count, config.COORDS), np.nan, dtype=np.float32)
    return np.array([(p.x, p.y, p.z) for p in landmarks.landmark], dtype=np.float32)


def results_to_frame(results) -> np.ndarray:
    return np.concatenate(
        [
            landmarks_to_array(results.pose_landmarks, config.NUM_POSE),
            landmarks_to_array(results.left_hand_landmarks, config.NUM_HAND),
            landmarks_to_array(results.right_hand_landmarks, config.NUM_HAND),
        ]
    )


def iter_frames(path: Path, start: int, end: int, stride: int = 1) -> Iterator[np.ndarray]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise OSError(f"cannot open {path}")
    try:
        if start > 0:
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        for offset in range(end - start):
            if offset % stride:
                if not cap.grab():
                    break
                continue
            ok, frame = cap.read()
            if not ok:
                break
            yield cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    finally:
        cap.release()


def create_holistic(model_complexity: int = config.HOLISTIC_MODEL_COMPLEXITY):
    import mediapipe as mp

    return mp.solutions.holistic.Holistic(
        static_image_mode=False,
        model_complexity=model_complexity,
        min_detection_confidence=config.HOLISTIC_MIN_DETECTION_CONFIDENCE,
        min_tracking_confidence=config.HOLISTIC_MIN_TRACKING_CONFIDENCE,
    )


def extract_video(holistic, path: Path, start: int, end: int, stride: int = 1) -> np.ndarray:
    frames = [
        results_to_frame(holistic.process(rgb)) for rgb in iter_frames(path, start, end, stride)
    ]
    if not frames:
        raise ValueError("no decodable frames")
    return np.stack(frames)


def detection_rates(keypoints: np.ndarray) -> tuple[float, float, float]:
    present = ~np.isnan(keypoints[..., 0])
    return tuple(
        float(present[:, part].all(axis=1).mean())
        for part in (
            slice(0, config.NUM_POSE),
            config.LEFT_HAND_SLICE,
            config.RIGHT_HAND_SLICE,
        )
    )


def save_atomic(array: np.ndarray, target: Path) -> None:
    tmp = target.with_name(f".{target.stem}.tmp.npy")
    np.save(tmp, array)
    os.replace(tmp, target)


_model_complexity = config.HOLISTIC_MODEL_COMPLEXITY
_frame_stride = config.FRAME_STRIDE


def _init_worker(model_complexity: int, frame_stride: int) -> None:
    global _model_complexity, _frame_stride
    _model_complexity = model_complexity
    _frame_stride = frame_stride
    warnings.filterwarnings("ignore", message="SymbolDatabase.GetPrototype", category=UserWarning)
    cv2.setNumThreads(1)


def _process(task: tuple[str, int, int, str]) -> ExtractionResult:
    video_id, start, end, out_dir = task
    began = time.perf_counter()
    try:
        # A fresh graph per video prevents tracking state leaking between clips.
        with create_holistic(_model_complexity) as holistic:
            keypoints = extract_video(
                holistic, config.VIDEO_DIR / f"{video_id}.mp4", start, end, _frame_stride
            )
        save_atomic(keypoints, Path(out_dir) / f"{video_id}.npy")
        pose, left, right = detection_rates(keypoints)
        return ExtractionResult(
            video_id, "ok", len(keypoints), pose, left, right, time.perf_counter() - began
        )
    except Exception as exc:  # recorded per video and reported
        return ExtractionResult(
            video_id, "failed", seconds=time.perf_counter() - began, error=repr(exc)
        )


def read_status(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=STATUS_FIELDS)
    return pd.read_csv(path, dtype={"video_id": str}, keep_default_na=False).drop_duplicates(
        "video_id", keep="last"
    )


def select_videos(manifest: pd.DataFrame, limit: int | None, split: str | None) -> pd.DataFrame:
    if split:
        manifest = manifest[manifest["split"] == split]
    if limit:
        manifest = manifest.sample(n=min(limit, len(manifest)), random_state=config.SEED)
    return manifest


def pending_tasks(
    selected: pd.DataFrame, status: pd.DataFrame, out_dir: Path, retry_failed: bool
) -> list[tuple[str, int, int, str]]:
    failed = set(status.loc[status["status"] == "failed", "video_id"])
    succeeded = set(status.loc[status["status"] == "ok", "video_id"])
    tasks = []
    for row in selected.itertuples():
        # An array without an "ok" status row was written after the run was interrupted.
        done = row.video_id in succeeded and (out_dir / f"{row.video_id}.npy").exists()
        if done or (row.video_id in failed and not retry_failed):
            continue
        tasks.append((row.video_id, int(row.clip_start), int(row.clip_end), str(out_dir)))
    return tasks


def process_memory_mb() -> float:
    proc = psutil.Process()
    rss = proc.memory_info().rss
    for child in proc.children(recursive=True):
        try:
            rss += child.memory_info().rss
        except psutil.NoSuchProcess:
            continue
    return rss / 2**20


@dataclass
class RunStats:
    peak_rss_mb: float
    mean_system_cpu_percent: float
    frames: int
    videos: int


def run(
    tasks: list,
    workers: int,
    status_path: Path,
    model_complexity: int = config.HOLISTIC_MODEL_COMPLEXITY,
    frame_stride: int = config.FRAME_STRIDE,
    max_minutes: float | None = None,
) -> RunStats:
    peak_mb = process_memory_mb()
    frames = done = 0
    began = time.perf_counter()
    psutil.cpu_percent(None)
    new_file = not status_path.exists()
    with (
        status_path.open("a", newline="") as handle,
        Pool(workers, _init_worker, (model_complexity, frame_stride)) as pool,
        tqdm(total=len(tasks), unit="video", mininterval=30, ascii=True) as progress,
    ):
        writer = csv.DictWriter(handle, fieldnames=STATUS_FIELDS)
        if new_file:
            writer.writeheader()
        for done, result in enumerate(pool.imap_unordered(_process, tasks), start=1):
            progress.update()
            writer.writerow(asdict(result))
            handle.flush()
            frames += result.frames
            peak_mb = max(peak_mb, process_memory_mb())
            if result.status == "failed":
                log.warning("failed %s: %s", result.video_id, result.error)
            if max_minutes and (time.perf_counter() - began) / 60 > max_minutes:
                # Leaving the Pool context terminates in-flight workers; resume picks them up.
                log.info("time limit of %.0f min reached after %d videos", max_minutes, done)
                break
            if done % 500 == 0 or done == len(tasks):
                rate = 60 * done / (time.perf_counter() - began)
                log.info(
                    "%d/%d videos, %.1f videos/min, ETA %.1f min, peak RSS %.0f MB",
                    done,
                    len(tasks),
                    rate,
                    (len(tasks) - done) / rate,
                    peak_mb,
                )
    return RunStats(peak_mb, psutil.cpu_percent(None), frames, done)


def summarise(status: pd.DataFrame, video_ids: set[str], stats: RunStats, wall: float) -> dict:
    subset = status[status["video_id"].isin(video_ids)]
    ok = subset[subset["status"] == "ok"].astype(
        {c: float for c in ("frames", "pose_rate", "left_rate", "right_rate", "seconds")}
    )
    total_frames = max(ok["frames"].sum(), 1)

    def frame_weighted(column: str) -> float:
        return round(float((ok[column] * ok["frames"]).sum() / total_frames), 4)

    return {
        "videos_selected": len(video_ids),
        "succeeded": int(len(ok)),
        "failed": int((subset["status"] == "failed").sum()),
        "failures": subset.loc[subset["status"] == "failed", ["video_id", "error"]].to_dict(
            "records"
        ),
        "frames_processed": int(ok["frames"].sum()),
        "pose_detection_rate": frame_weighted("pose_rate"),
        "left_hand_detection_rate": frame_weighted("left_rate"),
        "right_hand_detection_rate": frame_weighted("right_rate"),
        "videos_without_any_hand": int(((ok["left_rate"] == 0) & (ok["right_rate"] == 0)).sum()),
        "mean_seconds_per_video_per_worker": round(float(ok["seconds"].mean()), 3),
        "mean_ms_per_frame_per_worker": round(1000 * float(ok["seconds"].sum() / total_frames), 2),
        "wall_seconds_this_run": round(wall, 2),
        "videos_this_run": stats.videos,
        "videos_per_minute_this_run": round(60 * stats.videos / max(wall, 1e-9), 1),
        "frames_per_second_this_run": round(stats.frames / max(wall, 1e-9), 1),
        "mean_system_cpu_percent_this_run": stats.mean_system_cpu_percent,
        "peak_rss_mb_this_run": round(stats.peak_rss_mb, 1),
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--limit", type=int)
    parser.add_argument("--split", choices=config.SPLITS)
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--out-dir", type=Path, default=config.KEYPOINT_DIR)
    parser.add_argument("--retry-failed", action="store_true")
    parser.add_argument("--report", type=Path, default=config.REPORT_DIR / "extraction_report.json")
    parser.add_argument("--manifest", type=Path, default=config.TRAINING_MANIFEST_CSV)
    parser.add_argument(
        "--model-complexity", type=int, choices=(0, 1, 2), default=config.HOLISTIC_MODEL_COMPLEXITY
    )
    parser.add_argument("--frame-stride", type=int, default=config.FRAME_STRIDE)
    parser.add_argument("--max-minutes", type=float)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    os.environ.setdefault("GLOG_minloglevel", "2")

    args.out_dir.mkdir(parents=True, exist_ok=True)
    args.report.parent.mkdir(parents=True, exist_ok=True)
    status_path = args.out_dir / config.EXTRACTION_STATUS_CSV.name

    manifest = pd.read_csv(args.manifest, dtype={"video_id": str})
    selected = select_videos(manifest, args.limit, args.split)
    tasks = pending_tasks(selected, read_status(status_path), args.out_dir, args.retry_failed)
    log.info("%d selected, %d pending, %d workers", len(selected), len(tasks), args.workers)

    began = time.perf_counter()
    stats = (
        run(
            tasks,
            args.workers,
            status_path,
            args.model_complexity,
            args.frame_stride,
            args.max_minutes,
        )
        if tasks
        else RunStats(process_memory_mb(), 0.0, 0, 0)
    )
    report = summarise(
        read_status(status_path), set(selected["video_id"]), stats, time.perf_counter() - began
    )
    report["pending_at_start"] = len(tasks)
    report["config"] = {
        "manifest": str(args.manifest.name),
        "workers": args.workers,
        "model_complexity": args.model_complexity,
        "frame_stride": args.frame_stride,
    }
    args.report.write_text(json.dumps(report, indent=2))
    log.info(json.dumps({k: v for k, v in report.items() if k != "failures"}, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
def restore_keypoints() -> None:
    sources = [d for d in find_dirs("extraction_status.csv") if d.name == KEYPOINT_DIR.name]
    KEYPOINT_DIR.mkdir(parents=True, exist_ok=True)
    status_path = KEYPOINT_DIR / "extraction_status.csv"
    for source in sources:
        with (source / "extraction_status.csv").open() as handle:
            rows = [r for r in csv.DictReader(handle) if r["status"] == "ok"]
        copied = []
        for row in rows:
            target = KEYPOINT_DIR / f"{row['video_id']}.npy"
            if not target.exists() and (source / target.name).exists():
                shutil.copy2(source / target.name, target)
                copied.append(row)
        if copied:
            new_file = not status_path.exists()
            with status_path.open("a", newline="") as handle:
                writer = csv.DictWriter(handle, fieldnames=list(copied[0]))
                if new_file:
                    writer.writeheader()
                writer.writerows(copied)
        print(f"Restored {len(copied)} keypoint arrays from {source}")
    if not sources:
        print(f"No previous {KEYPOINT_DIR.name} keypoints attached; extracting from scratch.")


def restore_checkpoints() -> None:
    for source in find_dirs("last.pt"):
        if source.name == ARCH and not (CHECKPOINT_DIR / "last.pt").exists():
            CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
            for item in source.iterdir():
                shutil.copy2(item, CHECKPOINT_DIR / item.name)
            print(f"Restored training checkpoint from {source}")


restore_keypoints()
restore_checkpoints()
resources()

In [ ]:
reference_stride = read_json(PROJECT / "kaggle" / "reference.json")["frame_stride"]
if reference_stride == FRAME_STRIDE:
    run(MP_PY, "kaggle/kaggle_checks.py", "parity")
else:
    print("Reference keypoints were extracted at a different stride; parity check skipped.")

### 5.1 Small-sample tests (5 videos, resume, interruption) before full extraction

In [ ]:
SAMPLE_DIR = WORK / "sample_tests"


def sample_extract(limit: int, max_minutes: float | None = None) -> dict:
    command = [
        MP_PY,
        "-m",
        "src.extract_keypoints",
        "--limit",
        limit,
        "--workers",
        min(2, EXTRACT_WORKERS),
        "--out-dir",
        SAMPLE_DIR,
        "--report",
        SAMPLE_DIR / "report.json",
    ]
    if max_minutes:
        command += ["--max-minutes", max_minutes]
    run(*command)
    return read_json(SAMPLE_DIR / "report.json")


first = sample_extract(5)
assert first["succeeded"] == 5, first
run(MP_PY, "kaggle/kaggle_checks.py", "samples", SAMPLE_DIR)
assert sample_extract(5)["pending_at_start"] == 0, "resume did not skip finished videos"
assert sample_extract(8)["pending_at_start"] == 3, "resume did not extend the finished sample"
interrupted = sample_extract(14, max_minutes=0.05)
resumed = sample_extract(14)
assert resumed["succeeded"] == 14 and not list(SAMPLE_DIR.glob(".*tmp*")), resumed
print(
    f"Interruption test: stopped after {interrupted['videos_this_run']} new videos, "
    f"resumed {resumed['pending_at_start']} remaining, 14/14 complete, no temp files left."
)
print("Pre-extraction tests passed.")

### 5.2 Full extraction (time-limited; re-run the notebook to continue)

In [ ]:
extraction_minutes = minutes_left() - TRAINING_MINUTES - RESERVE_MINUTES
if RUN_FULL_EXTRACTION and extraction_minutes > 5:
    run(
        MP_PY,
        "-m",
        "src.extract_keypoints",
        "--workers",
        EXTRACT_WORKERS,
        "--max-minutes",
        round(extraction_minutes),
    )
else:
    print(
        f"Full extraction skipped (enabled={RUN_FULL_EXTRACTION}, "
        f"minutes available={extraction_minutes:.0f})."
    )
resources()

In [ ]:
manifest_name = "manifest.csv" if SUBSET == "full" else "reduced_manifest.csv"
manifest = pd.read_csv(PROJECT / "data" / "processed" / manifest_name, dtype={"video_id": str})
status = pd.read_csv(KEYPOINT_DIR / "extraction_status.csv", dtype={"video_id": str})
status = status.drop_duplicates("video_id", keep="last")
merged = manifest.merge(status[["video_id", "status"]], on="video_id", how="left")
merged["status"] = merged["status"].fillna("pending")
display(merged.groupby(["split", "status"]).size().unstack(fill_value=0))
failures = status[status["status"] == "failed"][["video_id", "error"]]
failures.to_csv(REPORT_DIR / "extraction_failures.csv", index=False)
print(f"Failures: {len(failures)} (listed in results/reports/extraction_failures.csv)")
EXTRACTION_COMPLETE = bool((merged["status"] != "pending").all())
if (REPORT_DIR / "extraction_report.json").exists():
    report = read_json(REPORT_DIR / "extraction_report.json")
    print({k: v for k, v in report.items() if k != "failures"})
print("Extraction complete:", EXTRACTION_COMPLETE)
if not EXTRACTION_COMPLETE:
    print("Save Version now, attach this version's output as input, and re-run to continue.")

## 6. Preprocessing

Per clip: short-gap interpolation, shoulder-centred and shoulder-width-scaled coordinates,
hips and legs removed, values clipped, uniform resampling to 64 frames, missing landmarks set
to 0. The same function is used by training, evaluation and inference; its configuration is
saved to `data/processed/preprocess_config.json`.

In [ ]:
%%writefile {PROJECT}/src/preprocess.py
import json
import logging
from dataclasses import asdict, dataclass, field

import numpy as np
import pandas as pd

from src import config

log = logging.getLogger("preprocess")

GROUPS = {
    "pose": slice(0, config.NUM_POSE),
    "left_hand": config.LEFT_HAND_SLICE,
    "right_hand": config.RIGHT_HAND_SLICE,
}


@dataclass
class PreprocessConfig:
    num_frames: int = config.NUM_FRAMES
    max_fill_gap: int = config.MAX_FILL_GAP
    clip_value: float = config.CLIP_VALUE
    frame_stride: int = config.FRAME_STRIDE
    dropped_pose_landmarks: list[int] = field(default_factory=lambda: list(config.DROPPED_POSE))
    landmark_order: list[str] = field(default_factory=lambda: list(config.LANDMARK_ORDER))

    def save(self, path) -> None:
        path.write_text(json.dumps(asdict(self), indent=2))

    @classmethod
    def load(cls, path) -> "PreprocessConfig":
        return cls(**json.loads(path.read_text()))


def _missing_runs(missing: np.ndarray) -> list[tuple[int, int]]:
    runs, start = [], None
    for t, flag in enumerate(missing):
        if flag and start is None:
            start = t
        elif not flag and start is not None:
            runs.append((start, t))
            start = None
    if start is not None:
        runs.append((start, len(missing)))
    return runs


def fill_missing(keypoints: np.ndarray, max_gap: int) -> np.ndarray:
    # Short interior gaps: linear interpolation. Short trailing gaps: hold last value.
    # Leading gaps and gaps longer than max_gap stay missing.
    out = keypoints.copy()
    frames = len(out)
    for part in GROUPS.values():
        missing = np.isnan(out[:, part, 0]).any(axis=1)
        for start, stop in _missing_runs(missing):
            if stop - start > max_gap or start == 0:
                continue
            before = out[start - 1, part]
            if stop == frames:
                out[start:stop, part] = before
                continue
            weights = (np.arange(start, stop) - (start - 1)) / (stop - (start - 1))
            after = out[stop, part]
            out[start:stop, part] = before + weights[:, None, None] * (after - before)
    return out


def shoulder_frame(keypoints: np.ndarray) -> tuple[np.ndarray, float]:
    left = keypoints[:, config.LEFT_SHOULDER]
    right = keypoints[:, config.RIGHT_SHOULDER]
    centre = (left + right) / 2
    width = np.linalg.norm(left[:, :2] - right[:, :2], axis=-1)
    valid = np.isfinite(width) & (width > config.MIN_SHOULDER_WIDTH)
    if not valid.any():
        return np.zeros(3, dtype=np.float32), 1.0
    # One clip-level reference keeps per-frame shoulder jitter out of the hand trajectories.
    return np.median(centre[valid], axis=0), float(np.median(width[valid]))


def normalise(keypoints: np.ndarray) -> np.ndarray:
    centre, width = shoulder_frame(keypoints)
    return (keypoints - centre) / width


def resample(keypoints: np.ndarray, num_frames: int) -> np.ndarray:
    # Linear interpolation between neighbouring frames; nearest frame where either is missing.
    positions = np.linspace(0, len(keypoints) - 1, num_frames)
    low = np.floor(positions).astype(int)
    high = np.minimum(low + 1, len(keypoints) - 1)
    weight = (positions - low)[:, None, None]
    blended = (1 - weight) * keypoints[low] + weight * keypoints[high]
    nearest = keypoints[positions.round().astype(int)]
    return np.where(np.isnan(blended), nearest, blended).astype(np.float32)


def preprocess(keypoints: np.ndarray, cfg: PreprocessConfig) -> np.ndarray:
    if keypoints.ndim != 3 or keypoints.shape[1:] != (config.NUM_LANDMARKS, config.COORDS):
        raise ValueError(f"expected (T, 75, 3), got {keypoints.shape}")
    if len(keypoints) == 0:
        raise ValueError("empty sequence")
    out = normalise(fill_missing(keypoints.astype(np.float32), cfg.max_fill_gap))
    out[:, cfg.dropped_pose_landmarks] = np.nan
    out = np.clip(out, -cfg.clip_value, cfg.clip_value)
    out = resample(out, cfg.num_frames)
    # Missing landmarks become exactly zero; the model reads all-zero groups as absent.
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def load_split(manifest: pd.DataFrame, ok_ids: set[str], split: str) -> tuple[list, pd.DataFrame]:
    rows = manifest[(manifest["split"] == split) & manifest["video_id"].isin(ok_ids)]
    arrays = [np.load(config.KEYPOINT_DIR / f"{vid}.npy") for vid in rows["video_id"]]
    return arrays, rows


def main() -> None:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    config.ensure_dirs()

    manifest = pd.read_csv(config.TRAINING_MANIFEST_CSV, dtype={"video_id": str})
    status = pd.read_csv(config.EXTRACTION_STATUS_CSV, dtype={"video_id": str})
    status = status.drop_duplicates("video_id", keep="last")
    ok_ids = set(status.loc[status["status"] == "ok", "video_id"])
    ok_ids &= {p.stem for p in config.KEYPOINT_DIR.glob("*.npy")}

    cfg = PreprocessConfig()
    cfg.save(config.PREPROCESS_CONFIG_JSON)
    report = {"config": asdict(cfg)}
    for split in config.SPLITS:
        arrays, rows = load_split(manifest, ok_ids, split)
        features = np.stack([preprocess(seq, cfg) for seq in arrays])
        labels = rows["label"].to_numpy(dtype=np.int64)
        np.savez(
            config.PROCESSED_DIR / f"{split}.npz",
            features=features,
            labels=labels,
            video_ids=rows["video_id"].to_numpy(),
            signer_ids=rows["signer_id"].to_numpy(),
        )
        selected = int((manifest["split"] == split).sum())
        report[split] = {
            "selected": selected,
            "processed": len(rows),
            "missing_or_failed": selected - len(rows),
            "classes": int(np.unique(labels).size),
            "shape": list(features.shape),
            "finite": bool(np.isfinite(features).all()),
        }
    (config.REPORT_DIR / "preprocess_report.json").write_text(json.dumps(report, indent=2))
    log.info(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
if EXTRACTION_COMPLETE:
    run(SYS_PY, "-m", "src.preprocess")
    print(json.dumps(read_json(REPORT_DIR / "preprocess_report.json"), indent=2))
    print(read_json(PROJECT / "data" / "processed" / "preprocess_config.json"))
else:
    print("Preprocessing waits for complete extraction.")

## 7. Augmentation (training only)

3D rotation (yaw +/-20 deg, roll +/-10 deg), scale and shift, mirroring with left/right
landmark swap, temporal crop and speed change, Gaussian jitter, one-hand occlusion for
10-40% of frames, random landmark dropout. Validation and test data are never augmented.

In [ ]:
%%writefile {PROJECT}/src/augment.py
import math

import torch

from src import config


def _mirror_index() -> list[int]:
    index = list(range(config.NUM_LANDMARKS))
    for a, b in config.POSE_MIRROR_PAIRS:
        index[a], index[b] = b, a
    left = range(config.LEFT_HAND_SLICE.start, config.LEFT_HAND_SLICE.stop)
    right = range(config.RIGHT_HAND_SLICE.start, config.RIGHT_HAND_SLICE.stop)
    for a, b in zip(left, right, strict=True):
        index[a], index[b] = b, a
    return index


MIRROR_INDEX = _mirror_index()


def presence_mask(x: torch.Tensor) -> torch.Tensor:
    return (x.abs().sum(dim=-1, keepdim=True) > 0).to(x.dtype)


def mirror(x: torch.Tensor) -> torch.Tensor:
    # Reflect across the body midline and swap left/right landmark identities.
    out = x[:, :, MIRROR_INDEX].clone()
    out[..., 0] = -out[..., 0]
    return out


def rotation_matrices(yaw: torch.Tensor, roll: torch.Tensor) -> torch.Tensor:
    cy, sy, cr, sr = yaw.cos(), yaw.sin(), roll.cos(), roll.sin()
    zeros, ones = torch.zeros_like(yaw), torch.ones_like(yaw)
    yaw_m = torch.stack([cy, zeros, sy, zeros, ones, zeros, -sy, zeros, cy], -1).view(-1, 3, 3)
    roll_m = torch.stack([cr, -sr, zeros, sr, cr, zeros, zeros, zeros, ones], -1).view(-1, 3, 3)
    return roll_m @ yaw_m


def rotate(x: torch.Tensor, yaw_deg: torch.Tensor, roll_deg: torch.Tensor) -> torch.Tensor:
    mask = presence_mask(x)
    rot = rotation_matrices(yaw_deg * math.pi / 180, roll_deg * math.pi / 180).to(x)
    return torch.einsum("btlc,bdc->btld", x, rot) * mask


def temporal_resample(x: torch.Tensor, generator: torch.Generator, min_keep: float) -> torch.Tensor:
    batch, frames = x.shape[:2]
    keep = min_keep + (1 - min_keep) * torch.rand(batch, generator=generator)
    start = (1 - keep) * torch.rand(batch, generator=generator)
    grid = torch.linspace(0, 1, frames)
    positions = (start[:, None] + keep[:, None] * grid) * (frames - 1)
    index = positions.round().long().clamp(0, frames - 1).to(x.device)
    return x[torch.arange(batch, device=x.device)[:, None], index]


def hide_hand(
    x: torch.Tensor, fraction: torch.Tensor, apply: torch.Tensor, generator: torch.Generator
) -> torch.Tensor:
    batch, frames = x.shape[:2]
    span = (fraction * frames).round().long()
    start = (torch.rand(batch, generator=generator) * (frames - span + 1)).long()
    t = torch.arange(frames)[None]
    hidden_frames = (t >= start[:, None]) & (t < (start + span)[:, None]) & apply[:, None]
    use_left = torch.rand(batch, generator=generator) < 0.5
    landmark = torch.zeros(batch, config.NUM_LANDMARKS, dtype=torch.bool)
    landmark[use_left, config.LEFT_HAND_SLICE] = True
    landmark[~use_left, config.RIGHT_HAND_SLICE] = True
    drop = (hidden_frames[:, :, None] & landmark[:, None]).to(x.device)
    return x.masked_fill(drop[..., None], 0.0)


def occlude(x: torch.Tensor, fraction: float, seed: int) -> torch.Tensor:
    generator = torch.Generator().manual_seed(seed)
    batch = x.shape[0]
    return hide_hand(
        x, torch.full((batch,), fraction), torch.ones(batch, dtype=torch.bool), generator
    )


def augment(x: torch.Tensor, generator: torch.Generator) -> torch.Tensor:
    batch = x.shape[0]

    def uniform(low: float, high: float, size=(batch,)) -> torch.Tensor:
        return low + (high - low) * torch.rand(size, generator=generator)

    x = temporal_resample(x, generator, min_keep=0.7)
    mask = presence_mask(x)
    flip = (torch.rand(batch, generator=generator) < 0.3).to(x.device)
    x = torch.where(flip[:, None, None, None], mirror(x), x)
    mask = torch.where(flip[:, None, None, None], mask[:, :, MIRROR_INDEX], mask)
    x = rotate(x, uniform(-20, 20), uniform(-10, 10))
    scale = uniform(0.9, 1.1).to(x)[:, None, None, None]
    shift = torch.cat([uniform(-0.05, 0.05, (batch, 2)), torch.zeros(batch, 1)], -1).to(x)
    x = x * scale + shift[:, None, None]
    x = x + 0.01 * torch.randn(x.shape, generator=generator).to(x)
    x = x * mask
    x = hide_hand(x, uniform(0.1, 0.4), torch.rand(batch, generator=generator) < 0.3, generator)
    dropout = (torch.rand(x.shape[:3], generator=generator) < 0.02).to(x.device)
    return x.masked_fill(dropout[..., None], 0.0)

In [ ]:
run(
    SYS_PY,
    "-c",
    """
import torch
from src import config
from src.augment import MIRROR_INDEX, augment, mirror, presence_mask
x = torch.randn(8, config.NUM_FRAMES, config.NUM_LANDMARKS, config.COORDS)
x[:, :20, config.LEFT_HAND_SLICE] = 0
out = augment(x, torch.Generator().manual_seed(0))
assert out.shape == x.shape and torch.isfinite(out).all()
assert torch.equal(mirror(mirror(x)), x) and sorted(MIRROR_INDEX) == list(range(75))
print('augment output', tuple(out.shape), '| mirror swaps hands and is an involution')
print('fraction of landmarks present before/after:',
      round(presence_mask(x).mean().item(), 3), round(presence_mask(out).mean().item(), 3))
print('augmentation is applied only inside the training loop (src/train.py)')
""",
)

## 8. Dataset and data loading

Each preprocessed split is held as one tensor on the training device and batched by index,
which avoids DataLoader worker processes; videos are never loaded into RAM.

In [ ]:
%%writefile {PROJECT}/src/dataset.py
from collections.abc import Iterator
from dataclasses import dataclass

import numpy as np
import torch

from src import config


@dataclass
class SplitData:
    features: torch.Tensor
    labels: torch.Tensor
    video_ids: np.ndarray

    def __len__(self) -> int:
        return len(self.labels)

    def to(self, device: torch.device) -> "SplitData":
        return SplitData(self.features.to(device), self.labels.to(device), self.video_ids)


def load_split(split: str) -> SplitData:
    with np.load(config.PROCESSED_DIR / f"{split}.npz", allow_pickle=True) as data:
        features = torch.from_numpy(data["features"].astype(np.float32))
        labels = torch.from_numpy(data["labels"].astype(np.int64))
        video_ids = data["video_ids"]
    expected = (config.NUM_FRAMES, config.NUM_LANDMARKS, config.COORDS)
    if tuple(features.shape[1:]) != expected:
        raise ValueError(f"{split}: expected (N, {expected}), got {tuple(features.shape)}")
    if not torch.isfinite(features).all():
        raise ValueError(f"{split}: non-finite features")
    return SplitData(features, labels, video_ids)


def check_disjoint(*splits: SplitData) -> None:
    seen: set[str] = set()
    for split in splits:
        ids = set(split.video_ids.tolist())
        if ids & seen:
            raise ValueError("video IDs shared between splits")
        seen |= ids


def batches(
    data: SplitData, batch_size: int, shuffle: bool, generator: torch.Generator | None = None
) -> Iterator[tuple[torch.Tensor, torch.Tensor]]:
    order = torch.randperm(len(data), generator=generator) if shuffle else torch.arange(len(data))
    order = order.to(data.labels.device)
    for start in range(0, len(data), batch_size):
        index = order[start : start + batch_size]
        yield data.features[index], data.labels[index]

In [ ]:
if EXTRACTION_COMPLETE:
    run(
        SYS_PY,
        "-c",
        """
from src.dataset import check_disjoint, load_split
splits = {name: load_split(name) for name in ('train', 'val', 'test')}
check_disjoint(*splits.values())
for name, data in splits.items():
    print(name, tuple(data.features.shape), 'classes', data.labels.unique().numel())
print('splits are disjoint by video ID')
""",
    )

## 9. Pure temporal 1D-CNN

Features per frame (computed in-graph after augmentation): 225 body coordinates +
126 wrist-relative upright handshape values + 225 velocities = 576. Network: Linear
576 -> 192, LayerNorm, ReLU, residual Conv1D (k=5), residual stride-2 Conv1D (k=5), each with
BatchNorm and ReLU, temporal average pooling, dropout, Linear -> 2,000. Feature
standardisation statistics are fitted on the training split only and stored in the model.

In [ ]:
%%writefile {PROJECT}/src/model.py
import logging

import torch
from torch import nn

from src import config

log = logging.getLogger("model")

HAND_SIZE_REF = 9
HAND_DIMS = config.NUM_HAND * config.COORDS
BODY_DIMS = config.NUM_LANDMARKS * config.COORDS
NUM_FEATURES_576 = BODY_DIMS + 2 * HAND_DIMS + BODY_DIMS


def landmark_presence(x: torch.Tensor) -> torch.Tensor:
    return (x.abs().sum(dim=-1) > 0).to(x.dtype)


def masked_velocity(values: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    velocity = (values[:, 1:] - values[:, :-1]) * valid[:, 1:] * valid[:, :-1]
    return torch.cat([torch.zeros_like(values[:, :1]), velocity], dim=1)


def hand_upright(hand: torch.Tensor) -> torch.Tensor:
    # Wrist-relative, rotated so wrist->middle knuckle points up (-y), unit length.
    relative = hand - hand[:, :, :1]
    axis = relative[:, :, HAND_SIZE_REF, :2]
    size = torch.linalg.vector_norm(axis, dim=-1, keepdim=True).clamp_min(1e-3)
    ux, uy = (axis / size).unbind(-1)
    x, y, z = relative.unbind(-1)
    rotated_x = -uy[..., None] * x + ux[..., None] * y
    rotated_y = -ux[..., None] * x - uy[..., None] * y
    upright = torch.stack([rotated_x, rotated_y, z], dim=-1) / size[..., None]
    return upright.clamp(-config.CLIP_VALUE, config.CLIP_VALUE).flatten(2)


def features_576(x: torch.Tensor) -> torch.Tensor:
    body = x.flatten(2)
    left, right = x[:, :, config.LEFT_HAND_SLICE], x[:, :, config.RIGHT_HAND_SLICE]
    hands = torch.cat([hand_upright(left), hand_upright(right)], dim=-1)
    valid = landmark_presence(x)[..., None].expand_as(x).flatten(2)
    return torch.cat([body, hands, masked_velocity(body, valid)], dim=-1)


class ConvBlock(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, kernel: int, stride: int = 1, act=nn.GELU):
        super().__init__()
        self.conv = nn.Conv1d(in_dim, out_dim, kernel, stride=stride, padding=kernel // 2)
        self.norm = nn.BatchNorm1d(out_dim)
        self.act = act()
        self.residual = stride == 1 and in_dim == out_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.act(self.norm(self.conv(x)))
        return x + out if self.residual else out


class ResidualDownBlock(ConvBlock):
    def __init__(self, dim: int, kernel: int):
        super().__init__(dim, dim, kernel, stride=2, act=nn.ReLU)
        self.shortcut = nn.AvgPool1d(2, ceil_mode=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.shortcut(x) + self.act(self.norm(self.conv(x)))


class FeatureStandardised(nn.Module):
    num_features: int

    def __init__(self) -> None:
        super().__init__()
        self.register_buffer("feature_mean", torch.zeros(self.num_features))
        self.register_buffer("feature_std", torch.ones(self.num_features))

    def features(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def set_feature_stats(self, mean: torch.Tensor, std: torch.Tensor) -> None:
        self.feature_mean.copy_(mean)
        self.feature_std.copy_(std.clamp_min(1e-4))

    def standardised_features(self, x: torch.Tensor) -> torch.Tensor:
        return (self.features(x) - self.feature_mean) / self.feature_std


class TemporalCNN(FeatureStandardised):
    """Baseline: (B, 64, 75, 3) landmarks -> (B, 64, 576) features -> 1D-CNN -> (B, 2000)."""

    num_features = NUM_FEATURES_576

    def __init__(
        self,
        num_classes: int = config.NUM_CLASSES,
        hidden: int = 192,
        kernel: int = 5,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(self.num_features, hidden), nn.LayerNorm(hidden), nn.ReLU()
        )
        self.blocks = nn.Sequential(
            ConvBlock(hidden, hidden, kernel, act=nn.ReLU), ResidualDownBlock(hidden, kernel)
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, num_classes))

    def features(self, x: torch.Tensor) -> torch.Tensor:
        return features_576(x)

    def classify_features(self, features: torch.Tensor) -> torch.Tensor:
        hidden = self.blocks(self.projection(features).transpose(1, 2))
        return self.head(hidden.mean(dim=-1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classify_features(self.standardised_features(x))


ARCHITECTURES = {"tcnn": TemporalCNN}


def build_model(arch: str) -> FeatureStandardised:
    if arch not in ARCHITECTURES:
        raise ValueError(f"unknown architecture {arch}; choose from {sorted(ARCHITECTURES)}")
    return ARCHITECTURES[arch]()


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def size_mb(model: nn.Module) -> float:
    tensors = [*model.parameters(), *model.buffers()]
    return sum(t.numel() * t.element_size() for t in tensors) / 2**20


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    sample = torch.randn(64, config.NUM_FRAMES, config.NUM_LANDMARKS, config.COORDS)
    for arch in ARCHITECTURES:
        net = build_model(arch).eval()
        log.info(
            "%s params=%d size=%.2f MB features=%s output=%s",
            arch,
            count_parameters(net),
            size_mb(net),
            tuple(net.features(sample).shape),
            tuple(net(sample).shape),
        )

In [ ]:
run(
    SYS_PY,
    "-c",
    """
import torch
from src import config
from src.model import build_model, count_parameters, size_mb
model = build_model(config.DEFAULT_ARCH).eval()
x = torch.randn(64, config.NUM_FRAMES, config.NUM_LANDMARKS, config.COORDS)
features = model.features(x)
logits = model(x)
print(model)
print('input', tuple(x.shape), '-> features', tuple(features.shape),
      '-> logits', tuple(logits.shape))
print(f'parameters {count_parameters(model):,} | size {size_mb(model):.2f} MB (FP32)')
assert features.shape[-1] == 576 and logits.shape == (64, config.NUM_CLASSES)
""",
)

## 10. Training

Cross-entropy with label smoothing 0.1, AdamW (lr 0.002, weight decay 0.05), OneCycleLR,
batch 64, up to 150 epochs, early stopping on validation macro F1. CUDA mixed precision
(bfloat16 where native, else float16 with loss scaling). `last.pt` is saved every epoch and
training resumes from it automatically; `best.pt` holds the best validation macro F1.

In [ ]:
%%writefile {PROJECT}/src/train.py
import argparse
import json
import logging
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
from sklearn.metrics import f1_score
from torch import nn

from src import config
from src.augment import augment
from src.dataset import SplitData, batches, check_disjoint, load_split
from src.model import build_model, count_parameters, size_mb

log = logging.getLogger("train")


@dataclass
class TrainConfig:
    arch: str = config.DEFAULT_ARCH
    epochs: int = 150
    batch_size: int = 64
    lr: float = 2e-3
    weight_decay: float = 0.05
    label_smoothing: float = 0.1
    patience: int = 20
    seed: int = config.SEED
    time_budget_min: float = 600.0
    amp: str = "auto"


def _usable(device: str) -> bool:
    # Some CUDA builds load but lack kernels for older GPUs; run a real op to confirm.
    try:
        (torch.ones(2, device=device) * 2).sum().item()
        return True
    except (RuntimeError, AssertionError):
        return False


def select_device() -> torch.device:
    requested = os.environ.get("ECHOGESTURE_DEVICE")
    if requested:
        return torch.device(requested)
    if torch.cuda.is_available() and _usable("cuda"):
        return torch.device("cuda")
    if torch.backends.mps.is_available() and _usable("mps"):
        return torch.device("mps")
    return torch.device("cpu")


def amp_dtype(device: torch.device, mode: str) -> torch.dtype | None:
    # Mixed precision only on CUDA: bfloat16 where supported, otherwise float16 with loss scaling.
    if mode == "off" or device.type != "cuda":
        return None
    try:
        native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        native_bf16 = torch.cuda.is_bf16_supported()
    return torch.bfloat16 if native_bf16 else torch.float16


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


@torch.no_grad()
def feature_stats(
    model: nn.Module, data: SplitData, batch_size: int = 512
) -> tuple[torch.Tensor, torch.Tensor]:
    # Fitted on clean (unaugmented) training data only.
    total, total_sq, count = 0.0, 0.0, 0
    for x, _ in batches(data, batch_size, shuffle=False):
        f = model.features(x).flatten(0, 1).cpu().double()
        total = total + f.sum(0)
        total_sq = total_sq + (f * f).sum(0)
        count += f.shape[0]
    mean = total / count
    std = (total_sq / count - mean * mean).clamp_min(0).sqrt()
    return mean.float(), std.float()


@torch.no_grad()
def predict_logits(model: nn.Module, data: SplitData, batch_size: int = 256) -> torch.Tensor:
    model.eval()
    return torch.cat([model(x).float().cpu() for x, _ in batches(data, batch_size, False)])


def classification_metrics(logits: torch.Tensor, labels: torch.Tensor) -> dict:
    labels = labels.cpu()
    loss = nn.functional.cross_entropy(logits, labels).item()
    top5 = logits.topk(5, dim=1).indices
    pred = top5[:, 0]
    return {
        "loss": loss,
        "top1": (pred == labels).float().mean().item(),
        "top5": (top5 == labels[:, None]).any(1).float().mean().item(),
        "macro_f1": f1_score(
            labels.numpy(),
            pred.numpy(),
            labels=np.unique(labels.numpy()),
            average="macro",
            zero_division=0,
        ),
    }


def save_atomic(state: dict, path: Path) -> None:
    tmp = path.with_suffix(".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)


def memory_mb(device: torch.device) -> float:
    rss = psutil.Process().memory_info().rss / 2**20
    if device.type == "mps":
        rss = max(rss, torch.mps.driver_allocated_memory() / 2**20)
    if device.type == "cuda":
        rss = max(rss, torch.cuda.max_memory_allocated() / 2**20)
    return rss


def train(cfg: TrainConfig, run_dir: Path, resume: bool) -> dict:
    seed_everything(cfg.seed)
    device = select_device()
    train_data, val_data = load_split("train"), load_split("val")
    check_disjoint(train_data, val_data, load_split("test"))
    train_data, val_data = train_data.to(device), val_data.to(device)

    model = build_model(cfg.arch).to(device)
    model.set_feature_stats(*[t.to(device) for t in feature_stats(model, train_data)])
    dtype = amp_dtype(device, cfg.amp)
    scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps_per_epoch = -(-len(train_data) // cfg.batch_size)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.lr, total_steps=cfg.epochs * steps_per_epoch, pct_start=0.1
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    generator = torch.Generator().manual_seed(cfg.seed)
    history, best_f1, best_epoch, start_epoch, elapsed = [], -1.0, -1, 0, 0.0

    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
    if resume and last_path.exists():
        state = torch.load(last_path, map_location=device, weights_only=False)
        saved = state["train_config"]
        if (saved["epochs"], saved["batch_size"], saved["arch"]) != (
            cfg.epochs,
            cfg.batch_size,
            cfg.arch,
        ):
            raise ValueError(f"resume needs the same epochs/batch_size/arch as {saved}")
        model.load_state_dict(state["model"])
        scaler.load_state_dict(state["scaler"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        generator.set_state(state["generator"])
        torch.set_rng_state(state["torch_rng"])
        history, best_f1, best_epoch = state["history"], state["best_f1"], state["best_epoch"]
        start_epoch, elapsed = state["epoch"] + 1, state["elapsed"]
        log.info("resumed at epoch %d", start_epoch)

    log.info(
        "device=%s amp=%s params=%d size=%.2fMB train=%d val=%d",
        device,
        dtype,
        count_parameters(model),
        size_mb(model),
        len(train_data),
        len(val_data),
    )
    peak_mb = memory_mb(device)
    began = time.perf_counter() - elapsed
    for epoch in range(start_epoch, cfg.epochs):
        model.train()
        epoch_began, running, seen = time.perf_counter(), 0.0, 0
        for x, y in batches(train_data, cfg.batch_size, True, generator):
            with torch.autocast(
                device.type, dtype=dtype or torch.float32, enabled=dtype is not None
            ):
                logits = model(augment(x, generator))
            loss = criterion(logits.float(), y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at epoch {epoch}")
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item() * len(y)
            seen += len(y)
        val = classification_metrics(predict_logits(model, val_data), val_data.labels)
        peak_mb = max(peak_mb, memory_mb(device))
        row = {
            "epoch": epoch,
            "train_loss": running / seen,
            "lr": scheduler.get_last_lr()[0],
            "epoch_seconds": time.perf_counter() - epoch_began,
        }
        row |= {f"val_{k}": v for k, v in val.items()}
        history.append(row)
        if val["macro_f1"] > best_f1:
            best_f1, best_epoch = val["macro_f1"], epoch
            save_atomic(
                {
                    "model": model.state_dict(),
                    "arch": cfg.arch,
                    "epoch": epoch,
                    "val": val,
                    "train_config": asdict(cfg),
                },
                best_path,
            )
        log.info(
            "epoch %3d loss %.3f val_loss %.3f top1 %.4f top5 %.4f f1 %.4f %.1fs",
            epoch,
            row["train_loss"],
            val["loss"],
            val["top1"],
            val["top5"],
            val["macro_f1"],
            row["epoch_seconds"],
        )
        save_atomic(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "train_config": asdict(cfg),
                "generator": generator.get_state(),
                "torch_rng": torch.get_rng_state(),
                "history": history,
                "best_f1": best_f1,
                "best_epoch": best_epoch,
                "epoch": epoch,
                "elapsed": time.perf_counter() - began,
            },
            last_path,
        )
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        if epoch - best_epoch >= cfg.patience:
            log.info("early stop: no val macro F1 gain for %d epochs", cfg.patience)
            break
        if (time.perf_counter() - began) / 60 > cfg.time_budget_min:
            log.info("time budget of %.0f min reached", cfg.time_budget_min)
            break

    best = history[best_epoch]
    return {
        "arch": cfg.arch,
        "device": str(device),
        "mixed_precision": str(dtype) if dtype else "off",
        "parameters": count_parameters(model),
        "model_size_mb": size_mb(model),
        "train_samples": len(train_data),
        "val_samples": len(val_data),
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_val": {k.removeprefix("val_"): v for k, v in best.items() if k.startswith("val_")},
        "final_train_loss": history[-1]["train_loss"],
        "train_loss_at_best": best["train_loss"],
        "mean_epoch_seconds": float(np.mean([h["epoch_seconds"] for h in history])),
        "training_minutes": (time.perf_counter() - began) / 60,
        "peak_memory_mb": peak_mb,
        "config": asdict(cfg),
    }


def plot_history(history_csv: Path, out: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    history = pd.read_csv(history_csv)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history["epoch"], history["train_loss"], label="train (augmented)")
    axes[0].plot(history["epoch"], history["val_loss"], label="val")
    axes[0].set_title("Loss")
    for key in ("val_top1", "val_top5", "val_macro_f1"):
        axes[1].plot(history["epoch"], history[key], label=key.removeprefix("val_"))
    axes[1].set_title("Validation")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.legend()
    fig.tight_layout()
    fig.savefig(out, dpi=120)
    plt.close(fig)


def main() -> None:
    parser = argparse.ArgumentParser()
    defaults = TrainConfig()
    for name, value in asdict(defaults).items():
        parser.add_argument(f"--{name.replace('_', '-')}", type=type(value), default=value)
    parser.add_argument("--resume", action="store_true")
    args = vars(parser.parse_args())
    resume = args.pop("resume")
    cfg = TrainConfig(**args)
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    config.ensure_dirs()

    run_dir = config.CHECKPOINT_DIR / cfg.arch
    run_dir.mkdir(parents=True, exist_ok=True)
    report = train(cfg, run_dir, resume)
    plot_history(run_dir / "history.csv", config.PLOT_DIR / f"training_{cfg.arch}.png")
    (config.REPORT_DIR / f"training_report_{cfg.arch}.json").write_text(
        json.dumps(report, indent=2)
    )
    log.info(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
run(SYS_PY, "-c", "from src.train import select_device; print('Training device:', select_device())")
if EXTRACTION_COMPLETE and RUN_TRAINING:
    budget = max(5, minutes_left() - RESERVE_MINUTES)
    command = [
        SYS_PY,
        "-m",
        "src.train",
        "--arch",
        ARCH,
        "--epochs",
        EPOCHS,
        "--batch-size",
        BATCH_SIZE,
        "--patience",
        PATIENCE,
        "--time-budget-min",
        round(budget),
    ]
    if (CHECKPOINT_DIR / "last.pt").exists():
        command.append("--resume")
    run(*command)
else:
    print("Training skipped until extraction is complete.")
resources()

## 11. Validation

In [ ]:
from IPython.display import Image

if (REPORT_DIR / f"training_report_{ARCH}.json").exists():
    training = read_json(REPORT_DIR / f"training_report_{ARCH}.json")
    print(
        json.dumps(
            {
                k: training[k]
                for k in (
                    "device",
                    "mixed_precision",
                    "parameters",
                    "model_size_mb",
                    "train_samples",
                    "val_samples",
                    "epochs_completed",
                    "best_epoch",
                    "best_val",
                    "training_minutes",
                    "peak_memory_mb",
                )
            },
            indent=2,
        )
    )
    display(Image(str(PROJECT / "results" / "plots" / f"training_{ARCH}.png")))

## 12. Test evaluation

Run once on the untouched official test split after model selection: top-1, top-5,
precision, recall, macro F1, per-class table, confusion matrix, occlusion (one hand hidden in
30% of frames) and rotation robustness. The official split is signer-dependent.

In [ ]:
%%writefile {PROJECT}/src/evaluate.py
import argparse
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

from src import config
from src.augment import occlude, rotate
from src.dataset import SplitData, load_split
from src.model import build_model
from src.train import classification_metrics, predict_logits, select_device

log = logging.getLogger("evaluate")

OCCLUSION_FRACTION = 0.3
ROTATION = {"yaw_deg": 20.0, "roll_deg": 10.0}


def load_checkpoint(path: Path, device: torch.device) -> torch.nn.Module:
    state = torch.load(path, map_location=device, weights_only=False)
    model = build_model(state["arch"]).to(device)
    model.load_state_dict(state["model"])
    return model.eval()


def full_metrics(logits: torch.Tensor, labels: torch.Tensor) -> dict:
    metrics = classification_metrics(logits, labels)
    y_true, y_pred = labels.cpu().numpy(), logits.argmax(1).numpy()
    precision, recall, _, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=np.unique(y_true), average="macro", zero_division=0
    )
    return metrics | {"macro_precision": float(precision), "macro_recall": float(recall)}


def per_class_table(logits: torch.Tensor, labels: torch.Tensor, glosses: dict) -> pd.DataFrame:
    y_true, y_pred = labels.cpu().numpy(), logits.argmax(1).numpy()
    classes = np.arange(config.NUM_CLASSES)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, zero_division=0
    )
    return pd.DataFrame(
        {
            "label": classes,
            "gloss": [glosses[c] for c in classes],
            "support": support,
            "correct": np.bincount(y_true[y_true == y_pred], minlength=config.NUM_CLASSES),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
    )


def top_confusions(logits: torch.Tensor, labels: torch.Tensor, glosses: dict) -> pd.DataFrame:
    y_true, y_pred = labels.cpu().numpy(), logits.argmax(1).numpy()
    wrong = y_true != y_pred
    pairs = pd.DataFrame({"true": y_true[wrong], "pred": y_pred[wrong]})
    counts = pairs.value_counts().rename("count").reset_index()
    counts["true_gloss"] = counts["true"].map(glosses)
    counts["pred_gloss"] = counts["pred"].map(glosses)
    return counts


def save_confusion(logits: torch.Tensor, labels: torch.Tensor, path: Path) -> None:
    matrix = confusion_matrix(
        labels.cpu().numpy(), logits.argmax(1).numpy(), labels=np.arange(config.NUM_CLASSES)
    )
    np.savez_compressed(path, confusion=matrix.astype(np.int32))


def variant(data: SplitData, features: torch.Tensor) -> SplitData:
    return SplitData(features, data.labels, data.video_ids)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--checkpoint", type=Path, default=config.CHECKPOINT_DIR / config.DEFAULT_ARCH / "best.pt"
    )
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    device = select_device()
    model = load_checkpoint(args.checkpoint, device)
    manifest = pd.read_csv(config.MANIFEST_CSV, dtype={"video_id": str})
    glosses = manifest.drop_duplicates("label").set_index("label")["gloss"].to_dict()

    rows, report = [], {"checkpoint": str(args.checkpoint)}
    for split in ("val", "test"):
        data = load_split(split).to(device)
        batch = len(data)
        variants = {
            "clean": data,
            "occluded": variant(
                data, occlude(data.features.cpu(), OCCLUSION_FRACTION, config.SEED).to(device)
            ),
            "rotated": variant(
                data,
                rotate(
                    data.features,
                    torch.full((batch,), ROTATION["yaw_deg"]),
                    torch.full((batch,), ROTATION["roll_deg"]),
                ),
            ),
        }
        results = {}
        for name, split_data in variants.items():
            logits = predict_logits(model, split_data)
            results[name] = full_metrics(logits, split_data.labels)
            rows.append(
                {"split": split, "variant": name, "samples": len(split_data)} | results[name]
            )
            if split == "test" and name == "clean":
                per_class_table(logits, data.labels, glosses).to_csv(
                    config.REPORT_DIR / "test_per_class.csv", index=False
                )
                save_confusion(logits, data.labels, config.REPORT_DIR / "test_confusion.npz")
                top_confusions(logits, data.labels, glosses).head(200).to_csv(
                    config.REPORT_DIR / "test_top_confusions.csv", index=False
                )
        for name in ("occluded", "rotated"):
            results[f"{name}_top1_drop_pp"] = 100 * (
                results["clean"]["top1"] - results[name]["top1"]
            )
        report[split] = results

    per_class = pd.read_csv(config.REPORT_DIR / "test_per_class.csv")
    tested = per_class[per_class["support"] > 0]
    report["test_per_class_summary"] = {
        "classes_tested": int(len(tested)),
        "classes_with_any_correct": int((tested["correct"] > 0).sum()),
        "median_class_recall": float(tested["recall"].median()),
    }
    report["robustness_settings"] = {"occluded_hand_fraction": OCCLUSION_FRACTION} | ROTATION
    report["note"] = (
        "Official WLASL split is signer-dependent: 99.97% of test videos come "
        "from signers also present in training."
    )
    pd.DataFrame(rows).to_csv(config.RESULTS_CSV, index=False)
    (config.REPORT_DIR / "evaluation_report.json").write_text(json.dumps(report, indent=2))
    log.info(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
if (CHECKPOINT_DIR / "best.pt").exists():
    run(SYS_PY, "-m", "src.evaluate", "--checkpoint", CHECKPOINT_DIR / "best.pt")
    display(pd.read_csv(PROJECT / "results" / "results.csv"))
    evaluation = read_json(REPORT_DIR / "evaluation_report.json")
    print(json.dumps({k: evaluation[k] for k in ("test_per_class_summary", "note")}, indent=2))

## 13. ONNX export

In [ ]:
%%writefile {PROJECT}/src/export_quantize.py
import argparse
import json
import logging
import time
from collections import Counter
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch
from onnxruntime.quantization import (
    CalibrationDataReader,
    QuantFormat,
    QuantType,
    quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process

from src import config
from src.dataset import load_split
from src.evaluate import load_checkpoint
from src.train import classification_metrics

log = logging.getLogger("export_quantize")

INPUT_NAME, OUTPUT_NAME = "keypoints", "logits"
CALIBRATION_CLIPS = 300
QAT_THRESHOLD_PP = 2.0
COMPUTE_OPS = {"Conv", "MatMul", "Gemm", "GRU", "LSTM"}
QDQ_OPS = {"QuantizeLinear", "DequantizeLinear"}


def export_onnx(model: torch.nn.Module, path: Path) -> None:
    dummy = torch.zeros(1, config.NUM_FRAMES, config.NUM_LANDMARKS, config.COORDS)
    kwargs = {
        "input_names": [INPUT_NAME],
        "output_names": [OUTPUT_NAME],
        "opset_version": 17,
        "dynamic_axes": {INPUT_NAME: {0: "batch"}, OUTPUT_NAME: {0: "batch"}},
    }
    try:
        torch.onnx.export(model.cpu().eval(), (dummy,), str(path), dynamo=False, **kwargs)
    except TypeError:
        # PyTorch < 2.5 has no dynamo argument and always uses the TorchScript exporter.
        torch.onnx.export(model.cpu().eval(), (dummy,), str(path), **kwargs)
    onnx.checker.check_model(onnx.load(str(path)))


def session(path: Path) -> ort.InferenceSession:
    return ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])


def run_onnx(sess: ort.InferenceSession, features: np.ndarray, batch: int = 256) -> np.ndarray:
    outputs = [
        sess.run([OUTPUT_NAME], {INPUT_NAME: features[i : i + batch]})[0]
        for i in range(0, len(features), batch)
    ]
    return np.concatenate(outputs)


def latency_ms(sess: ort.InferenceSession, sample: np.ndarray, warmup=20, runs=200) -> dict:
    for _ in range(warmup):
        sess.run([OUTPUT_NAME], {INPUT_NAME: sample})
    times = []
    for _ in range(runs):
        began = time.perf_counter()
        sess.run([OUTPUT_NAME], {INPUT_NAME: sample})
        times.append(1000 * (time.perf_counter() - began))
    return {
        "p50": float(np.percentile(times, 50)),
        "p95": float(np.percentile(times, 95)),
        "mean": float(np.mean(times)),
        "warmup": warmup,
        "runs": runs,
    }


class TrainCalibration(CalibrationDataReader):
    def __init__(self, features: np.ndarray):
        self._iter = iter(features[:, None])

    def get_next(self):
        sample = next(self._iter, None)
        return None if sample is None else {INPUT_NAME: sample}


def quantize_int8(fp32_path: Path, int8_path: Path, calibration: np.ndarray) -> dict:
    prepared = int8_path.with_name("model_fp32_prep.onnx")
    quant_pre_process(str(fp32_path), str(prepared))
    quantize_static(
        str(prepared),
        str(int8_path),
        TrainCalibration(calibration),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    prepared.unlink()
    return operator_precision(onnx.load(str(int8_path)))


def operator_precision(model: onnx.ModelProto) -> dict:
    # A compute node runs in INT8 only when its weight input is produced by DequantizeLinear.
    nodes = model.graph.node
    dequantized = {out for n in nodes if n.op_type == "DequantizeLinear" for out in n.output}
    int8, fp32 = Counter(), Counter()
    for node in nodes:
        if node.op_type in COMPUTE_OPS:
            weight = node.input[1] if len(node.input) > 1 else ""
            (int8 if weight in dequantized else fp32)[node.op_type] += 1
    other = Counter(n.op_type for n in nodes if n.op_type not in COMPUTE_OPS | QDQ_OPS)
    return {
        "int8_compute_ops": dict(int8),
        "fp32_compute_ops": dict(fp32),
        "fp32_other_ops": dict(other),
        "fully_int8_compute": not fp32,
        "quantize_linear_nodes": sum(n.op_type == "QuantizeLinear" for n in nodes),
    }


def compare(reference: np.ndarray, candidate: np.ndarray) -> dict:
    return {
        "max_abs_logit_diff": float(np.abs(reference - candidate).max()),
        "argmax_agreement": float((reference.argmax(1) == candidate.argmax(1)).mean()),
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--checkpoint", type=Path, default=config.CHECKPOINT_DIR / config.DEFAULT_ARCH / "best.pt"
    )
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    config.ensure_dirs()
    torch.set_num_threads(4)

    model = load_checkpoint(args.checkpoint, torch.device("cpu"))
    fp32_path, int8_path = (
        config.EXPORT_DIR / "model_fp32.onnx",
        config.EXPORT_DIR / "model_int8.onnx",
    )
    export_onnx(model, fp32_path)

    train = load_split("train")
    rng = np.random.default_rng(config.SEED)
    # Calibration draws from the training split only.
    calibration = train.features.numpy()[rng.choice(len(train), CALIBRATION_CLIPS, replace=False)]
    report = {"int8_quantization": quantize_int8(fp32_path, int8_path, calibration)}

    fp32, int8 = session(fp32_path), session(int8_path)
    sample = load_split("val").features.numpy()[:1]
    for split in ("val", "test"):
        data = load_split(split)
        x = data.features.numpy()
        with torch.no_grad():
            torch_logits = torch.cat(
                [model(data.features[i : i + 256]) for i in range(0, len(data), 256)]
            ).numpy()
        fp32_logits, int8_logits = run_onnx(fp32, x), run_onnx(int8, x)
        report[split] = {
            "torch_fp32": classification_metrics(torch.from_numpy(torch_logits), data.labels),
            "onnx_fp32": classification_metrics(torch.from_numpy(fp32_logits), data.labels),
            "onnx_int8": classification_metrics(torch.from_numpy(int8_logits), data.labels),
            "onnx_fp32_vs_torch": compare(torch_logits, fp32_logits),
            "onnx_int8_vs_fp32": compare(fp32_logits, int8_logits),
        }
    drop_pp = 100 * (report["val"]["onnx_fp32"]["top1"] - report["val"]["onnx_int8"]["top1"])
    report["int8_val_top1_drop_pp"] = drop_pp
    report["qat_recommended"] = drop_pp > QAT_THRESHOLD_PP
    report["latency_batch1_ms"] = {
        "onnx_fp32": latency_ms(fp32, sample),
        "onnx_int8": latency_ms(int8, sample),
    }
    report["size_mb"] = {
        "onnx_fp32": fp32_path.stat().st_size / 2**20,
        "onnx_int8": int8_path.stat().st_size / 2**20,
    }
    report["runtime"] = {
        "onnxruntime": ort.__version__,
        "provider": "CPUExecutionProvider",
        "hardware": "Apple M4 MacBook Air (CPU)",
    }
    report["input"] = {
        "name": INPUT_NAME,
        "shape": ["batch", config.NUM_FRAMES, config.NUM_LANDMARKS, config.COORDS],
    }
    (config.REPORT_DIR / "export_report.json").write_text(json.dumps(report, indent=2))
    log.info(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
if (CHECKPOINT_DIR / "best.pt").exists():
    run(SYS_PY, "-m", "src.export_quantize", "--checkpoint", CHECKPOINT_DIR / "best.pt")
    export = read_json(REPORT_DIR / "export_report.json")
    for split in ("val", "test"):
        print(split, "ONNX FP32 vs PyTorch:", export[split]["onnx_fp32_vs_torch"])
    print("FP32 size MB:", round(export["size_mb"]["onnx_fp32"], 2))

## 14. INT8 quantization

Static post-training QDQ quantization calibrated on 300 training clips (never test data).
The report lists exactly which operators run in INT8 and which remain FP32.

In [ ]:
if (REPORT_DIR / "export_report.json").exists():
    export = read_json(REPORT_DIR / "export_report.json")
    print("Operator precision:", json.dumps(export["int8_quantization"], indent=2))
    rows = []
    for split in ("val", "test"):
        for variant in ("torch_fp32", "onnx_fp32", "onnx_int8"):
            rows.append({"split": split, "model": variant} | export[split][variant])
    display(pd.DataFrame(rows))
    print("INT8 vs FP32 agreement:", export["test"]["onnx_int8_vs_fp32"])
    print(
        f"Validation top-1 drop from INT8: {export['int8_val_top1_drop_pp']:.2f} pp "
        f"| QAT recommended: {export['qat_recommended']}"
    )
    print("Sizes MB:", {k: round(v, 2) for k, v in export["size_mb"].items()})
    pd.DataFrame(rows).to_csv(REPORT_DIR / "quantization_comparison.csv", index=False)

## 15. Benchmarking and local inference

Model-only latency (20 warm-up + 200 runs, batch 1) comes from the export report. The
end-to-end benchmark separates MediaPipe landmark time, preprocessing and model time on test
videos. `main.py` shows the JSON returned by `predict(video_path)`.

In [ ]:
%%writefile {PROJECT}/src/inference.py
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd

from src import config
from src.extract_keypoints import create_holistic, extract_video
from src.preprocess import PreprocessConfig, preprocess

DEFAULT_MODEL = config.EXPORT_DIR / "model_fp32.onnx"


def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = np.exp(logits - logits.max())
    return shifted / shifted.sum()


@dataclass
class Recognizer:
    model_path: Path = DEFAULT_MODEL
    preprocess_config: Path = config.PREPROCESS_CONFIG_JSON
    top_k: int = 5

    def __post_init__(self) -> None:
        self.cfg = PreprocessConfig.load(self.preprocess_config)
        self.session = ort.InferenceSession(
            str(self.model_path), providers=["CPUExecutionProvider"]
        )
        frames = self.session.get_inputs()[0].shape[1]
        if frames != self.cfg.num_frames:
            raise ValueError(
                f"model expects {frames} frames but preprocessing produces {self.cfg.num_frames}"
            )
        manifest = pd.read_csv(config.MANIFEST_CSV, usecols=["label", "gloss"])
        self.glosses = manifest.drop_duplicates("label").set_index("label")["gloss"].to_dict()

    def classify(self, keypoints: np.ndarray) -> tuple[list[dict], dict]:
        began = time.perf_counter()
        features = preprocess(keypoints, self.cfg)[None]
        prepared = time.perf_counter()
        logits = self.session.run(None, {"keypoints": features})[0][0]
        finished = time.perf_counter()
        probs = softmax(logits)
        best = np.argsort(probs)[::-1][: self.top_k]
        predictions = [
            {"word": self.glosses[int(i)].upper(), "label": int(i), "confidence": float(probs[i])}
            for i in best
        ]
        timing = {
            "preprocess_ms": 1000 * (prepared - began),
            "model_ms": 1000 * (finished - prepared),
        }
        return predictions, timing

    def extract(self, video_path: Path) -> tuple[np.ndarray, float]:
        began = time.perf_counter()
        frames = _frame_count(video_path)
        with create_holistic() as holistic:
            keypoints = extract_video(holistic, video_path, 0, frames, self.cfg.frame_stride)
        return keypoints, 1000 * (time.perf_counter() - began)

    def predict_video(self, video_path: Path) -> dict:
        keypoints, landmarks_ms = self.extract(video_path)
        predictions, timing = self.classify(keypoints)
        return {
            "word": predictions[0]["word"],
            "confidence": predictions[0]["confidence"],
            "top5": predictions,
            "latency_ms": {"landmarks_ms": landmarks_ms}
            | timing
            | {"end_to_end_ms": landmarks_ms + sum(timing.values())},
            "frames_used": int(len(keypoints)),
        }


def _frame_count(video_path: Path) -> int:
    import cv2

    cap = cv2.VideoCapture(str(video_path))
    try:
        if not cap.isOpened():
            raise OSError(f"cannot open {video_path}")
        return int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    finally:
        cap.release()

In [ ]:
%%writefile {PROJECT}/src/benchmark.py
import argparse
import json
import logging
import platform

import numpy as np
import pandas as pd

from src import config
from src.inference import Recognizer

log = logging.getLogger("benchmark")


def percentiles(values: list[float]) -> dict:
    return {
        "p50": float(np.percentile(values, 50)),
        "p95": float(np.percentile(values, 95)),
        "mean": float(np.mean(values)),
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--videos", type=int, default=30)
    parser.add_argument("--warmup", type=int, default=3)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

    models = {
        "onnx_fp32": config.EXPORT_DIR / "model_fp32.onnx",
        "onnx_int8": config.EXPORT_DIR / "model_int8.onnx",
    }
    recognizers = {name: Recognizer(path) for name, path in models.items() if path.exists()}
    manifest = pd.read_csv(config.MANIFEST_CSV, dtype={"video_id": str})
    test = manifest[manifest["split"] == "test"].sample(
        n=args.videos + args.warmup, random_state=config.SEED
    )
    timings: dict[str, list[float]] = {"landmarks_ms": []}
    extractor = next(iter(recognizers.values()))
    for index, video_id in enumerate(test["video_id"]):
        keypoints, landmarks_ms = extractor.extract(config.VIDEO_DIR / f"{video_id}.mp4")
        for name, recognizer in recognizers.items():
            _, timing = recognizer.classify(keypoints)
            if index >= args.warmup:
                for key, value in timing.items():
                    timings.setdefault(f"{name}_{key}", []).append(value)
                timings.setdefault(f"{name}_end_to_end_ms", []).append(
                    landmarks_ms + sum(timing.values())
                )
        if index >= args.warmup:
            timings["landmarks_ms"].append(landmarks_ms)
    report = {
        "videos": args.videos,
        "warmup": args.warmup,
        "frame_stride": config.FRAME_STRIDE,
        "platform": f"{platform.system()} {platform.machine()} {platform.processor()}",
        "latency_ms": {key: percentiles(values) for key, values in timings.items()},
    }
    (config.REPORT_DIR / "benchmark_report.json").write_text(json.dumps(report, indent=2))
    log.info(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {PROJECT}/main.py
import argparse
import json
from functools import lru_cache
from pathlib import Path

from src.inference import DEFAULT_MODEL, Recognizer


@lru_cache(maxsize=2)
def _recognizer(model_path: str) -> Recognizer:
    return Recognizer(Path(model_path))


def predict(video_path: str | Path, model_path: str | Path = DEFAULT_MODEL) -> dict:
    return _recognizer(str(model_path)).predict_video(Path(video_path))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("video", type=Path)
    parser.add_argument("--model", type=Path, default=DEFAULT_MODEL)
    args = parser.parse_args()
    print(json.dumps(predict(args.video, args.model), indent=2))

In [ ]:
if (PROJECT / "models" / "exported" / "model_fp32.onnx").exists():
    export = read_json(REPORT_DIR / "export_report.json")
    print("Model-only latency, batch 1 (ms):", json.dumps(export["latency_batch1_ms"], indent=2))
    run(MP_PY, "-m", "src.benchmark", "--videos", BENCHMARK_VIDEOS)
    test_video = RAW_DIR / "videos" / f"{manifest[manifest.split == 'test'].video_id.iloc[0]}.mp4"
    run(MP_PY, "main.py", test_video)

## 16. Save downloadable artifacts

Collects checkpoints, ONNX models, preprocessing config, class mapping, reports, plots and
frozen requirements in `/kaggle/working/echogesture_output` and `echogesture_output.zip`.
`/kaggle/working` does not survive the session: use **Save Version -> Save & Run All** to keep
it as notebook output (see KAGGLE_README.md).

In [ ]:
def copy_into_output(relative: str) -> None:
    source = PROJECT / relative
    if not source.exists():
        print(f"missing, not saved: {relative}")
        return
    target = OUTPUT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(
            source, target, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__")
        )
    else:
        shutil.copy2(source, target)


for relative in (
    f"models/checkpoints/{ARCH}/best.pt",
    f"models/checkpoints/{ARCH}/history.csv",
    "models/exported/model_fp32.onnx",
    "models/exported/model_int8.onnx",
    "data/processed/preprocess_config.json",
    "data/processed/manifest.csv",
    "data/processed/reduced_manifest.csv",
    "data/processed/excluded_samples.csv",
    f"data/keypoints/{KEYPOINT_DIR.name}/extraction_status.csv",
    "results",
    "src",
    "kaggle",
    "main.py",
):
    copy_into_output(relative)

full = pd.read_csv(PROJECT / "data" / "processed" / "manifest.csv")
class_map = full.drop_duplicates("label").sort_values("label")
(OUTPUT / "class_mapping.json").write_text(
    json.dumps({int(r.label): r.gloss for r in class_map.itertuples()}, indent=2)
)
for name, python in (("training", SYS_PY), ("extraction", MP_PY)):
    frozen = subprocess.run([str(python), "-m", "pip", "freeze"], capture_output=True, text=True)
    if frozen.returncode:
        frozen = subprocess.run(
            [str(SYS_PY), "-m", "uv", "pip", "freeze", "--python", str(python)],
            capture_output=True,
            text=True,
        )
    (OUTPUT / f"requirements-{name}-frozen.txt").write_text(frozen.stdout)
archive = shutil.make_archive(str(WORK / "echogesture_output"), "zip", OUTPUT)
print(f"Saved {archive} ({Path(archive).stat().st_size / 2**20:.1f} MB)")
for path in sorted(OUTPUT.rglob("*")):
    if path.is_file() and "src" not in path.parts and "kaggle" not in path.parts:
        print(f"  {path.relative_to(OUTPUT)}  {path.stat().st_size / 2**20:.2f} MB")
resources()